# GPT-OSS-120B native-tool lifecycle smoke
Private development only. One game, 60 seconds after model readiness. No competition submission.


In [ ]:
import json
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

# True only inside a real competition rerun; switches diagnostics + soft deadline.
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()

# Non-interactive matplotlib backend: diagnostics render plots with no display attached.
os.environ["MPLBACKEND"] = "Agg"
# Marks the run as a (real or emulated) submission so the framework + solver can adjust.
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
# Skip periodic JSON/HTML diagnostics and per-frame logging for every run.
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1"

# Gate 2 imports the analyzer at the control setting; each sequential trial then sets its own cap and paired seed before constructing agents.
os.environ["LOCAL_ANALYZER_MAX_OUTPUT"] = "0"
os.environ["LOCAL_ANALYZER_SEED"] = "-1"

# Model identity is checked by the offline runtime helper.
if TRUE_SUBMISSION:
    raise RuntimeError("Private GPT-OSS smoke must never be submitted")
os.environ["MULTIMODAL_CONTEXT"] = ""
# Pin arc_agi's cached level_reset_only before its client is built (RESET keeps the level).
os.environ["ONLY_RESET_LEVELS"] = "true"

# Prepend the CUDA toolkit to the linker path (it is off it on Kaggle GPU images) so the
# solver's GPU libraries (e.g. vllm / torch) can link against libcuda.
cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry for entry in [cuda_library_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if entry
)

# Everything the run produces is written here.
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print(f"taaf.kaggle: TRUE_SUBMISSION={TRUE_SUBMISSION}")

In [ ]:
# Install the ARC runtime from the bundled competition wheels.
# Quiet: stdout is discarded; stderr (and a non-zero exit) still surface real failures.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels",
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)

In [ ]:
# Kaggle inputs attached to this notebook, plus bookkeeping paths used below.
DATASET_SOURCES = ["keithtyser/duck-qwen38-nvfp4-mtp-vllm-smoke-v1", "keithtyser/qwen38-flash-next-vllm-nvfp4-runtime-v1"]
KERNEL_SOURCES = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"


# Locate the source dataset by its marker file rather than a fixed mount path.
def _find_bundle_dir() -> Path:
    for marker in Path("/kaggle/input").rglob(DATASET_BUNDLE_MARKER):
        return marker.parent
    raise RuntimeError("TAAF source bundle not found under /kaggle/input.")


# Kaggle mounts a dataset at /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>
# (depending on owner / slug collisions), so probe both and use whichever exists. Utility
# scripts mount under /kaggle/usr/lib/notebooks/<owner>/<slug>.
def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((c for c in candidates if c.exists()), None)


BUNDLE_DIR = _find_bundle_dir()
print(f"taaf.kaggle: source bundle = {BUNDLE_DIR}")

# Map each attached input to where Kaggle actually mounted it (the source bundle is index 0).
kaggle_input_paths: dict[str, str] = {}
for i, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if i == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# Published to setup commands and the solver via the environment:
setup_env = {
    # JSON {ref: mount_path} so they can locate every attached dataset / utility script.
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    # The attached dataset refs in order (index 0 is this source bundle).
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    # The attached utility-script / kernel refs.
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n")
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")

In [ ]:
# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


# Environment handed to each setup command (paths + any keys it has persisted).
def _command_env() -> dict:
    env = os.environ.copy()
    # "$PYTHON" in a command resolves to this notebook's interpreter.
    env["PYTHON"] = sys.executable
    # Absolute path to the mounted source bundle.
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    # The writable /kaggle/working directory.
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    # A command writes a JSON object here to persist env keys to later commands + the run.
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries))
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)")


# Harmony's Rust tokenizer has its own cache, separate from Hugging Face.
import gzip as _vocab_gzip, hashlib as _vocab_hashlib
_gptoss_vocab_input_paths = _dataset_mount_candidates('anandsingh8687/arc3-harmony-vocab-20260917')
_gptoss_vocab_source = next((p / name for p in _gptoss_vocab_input_paths
                            for name in ("o200k_base.tiktoken", "o200k_base.tiktoken.gz")
                            if (p / name).is_file()), None)
if _gptoss_vocab_source is None:
    raise RuntimeError("Pinned offline Harmony vocabulary input is missing")
_gptoss_vocab_bytes = _gptoss_vocab_source.read_bytes()
if _gptoss_vocab_source.suffix == ".gz":
    _gptoss_vocab_bytes = _vocab_gzip.decompress(_gptoss_vocab_bytes)
if (len(_gptoss_vocab_bytes) != 3613922 or
    _vocab_hashlib.sha256(_gptoss_vocab_bytes).hexdigest() != '446a9538cb6c348e3516120d7c08b09f57c36495e2acfffe59a5bf8b0cfb1a2d'):
    raise RuntimeError("Offline Harmony vocabulary failed checksum")
_gptoss_vocab_dir = WORKING_DIR / "gptoss-harmony-cache"
_gptoss_vocab_dir.mkdir(parents=True, exist_ok=True)
(_gptoss_vocab_dir / 'fb374d419588a4632f3f557e76b4b70aebbca790').write_bytes(_gptoss_vocab_bytes)
del _gptoss_vocab_bytes

# Reuse the pinned offline runtime extraction, not Qwen's model launcher.
_gptoss_module_dir = WORKING_DIR / "gptoss-runtime"
_gptoss_module_dir.mkdir(exist_ok=True)
(_gptoss_module_dir / "gptoss_runtime.py").write_text('"""Bounded offline lifecycle helper for the Kaggle GPT-OSS-120B candidate.\n\nThe attached runtime was originally prepared for another model.  This module\nuses only its pinned *resolver* and *extractor*, then derives a clean GPT-OSS\nenvironment from the verified filesystem.  In particular, it never invokes\nthe Qwen PLE patch, runtime-environment, probe, or main functions.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport importlib.util\nimport json\nimport os\nimport signal\nimport socket\nimport subprocess\nimport sys\nimport threading\nimport time\nimport urllib.error\nimport urllib.request\nfrom pathlib import Path\nfrom typing import Any\n\n\nHOST = "127.0.0.1"\nPORT = 1234\nBASE_URL = f"http://{HOST}:{PORT}/v1"\nMODEL_NAME = "gpt-oss-120b"\nMODEL_DATASET = "danielhanchen/gpt-oss-120b"\nRUNTIME_DATASET = "keithtyser/qwen38-flash-next-vllm-nvfp4-runtime-v1"\nBOOTSTRAP_SHA256 = "037c041c9bd9dcffa9084b32f47af9cf1bf35849d5eaf2e1a3422daac098b2e2"\nBOOTSTRAP_RUNTIME_MANIFEST_SHA256 = (\n    "e9453f8d0e9c5eb2e14712e0f8563aaa96752ddc1705f245cac327537502baad"\n)\nBOOTSTRAP_VLLM_VERSION = "0.1.dev20073+g8e685d198"\nMODEL_TYPE = "gpt_oss"\nMAX_MODEL_LEN = 32_768\nMAX_NUM_SEQS = 16\nGPU_MEMORY_UTILIZATION = "0.90"\nMETRICS_TIMEOUT_SECONDS = 10.0\nTERM_GRACE_SECONDS = 30.0\nKILL_GRACE_SECONDS = 15.0\nMAX_WATCHDOG_SECONDS = 20 * 60\n\nIDENTITY_FILENAME = "gptoss-server-identity.json"\nPROVENANCE_FILENAME = "gptoss-runtime-provenance.json"\nLOG_FILENAME = "gptoss-vllm.log"\nMETRICS_FILENAME = "gptoss-vllm-metrics-final.prom"\nTEARDOWN_FILENAME = "gptoss-server-teardown.json"\nTOOL_SMOKE_FILENAME = "gptoss-tool-smoke.json"\nHARMONY_PREFLIGHT_FILENAME = "gptoss-harmony-preflight.json"\nSTARTUP_FAILURE_FILENAME = "gptoss-startup-failure.json"\nHARMONY_CACHE_DIRNAME = "gptoss-harmony-cache"\n# openai-harmony 0.0.8 resolves the GPT-OSS vocabulary through tiktoken-rs\n# using this content-addressed cache key.  The notebook builder places this\n# exact official blob in the working directory before startup.\nHARMONY_TIKTOKEN_CACHE_KEY = "fb374d419588a4632f3f557e76b4b70aebbca790"\nHARMONY_VOCAB_BYTES = 3_613_922\nHARMONY_VOCAB_SHA256 = "446a9538cb6c348e3516120d7c08b09f57c36495e2acfffe59a5bf8b0cfb1a2d"\nHARMONY_PREFLIGHT_MAX_SECONDS = 20.0\n_ATEXIT_GUARDS: set[str] = set()\n\n\ndef _json_write(path: Path, value: dict[str, Any]) -> None:\n    temporary = path.with_name(f".{path.name}.tmp")\n    temporary.write_text(json.dumps(value, indent=2, sort_keys=True) + "\\n", encoding="utf-8")\n    temporary.replace(path)\n\n\ndef _json_read(path: Path) -> dict[str, Any]:\n    value = json.loads(path.read_text(encoding="utf-8"))\n    if not isinstance(value, dict):\n        raise RuntimeError(f"Expected JSON object in {path}.")\n    return value\n\n\ndef _sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef _boot_id() -> str:\n    return Path("/proc/sys/kernel/random/boot_id").read_text(encoding="utf-8").strip()\n\n\ndef _proc_record(pid: int) -> dict[str, Any] | None:\n    """Return a PID-reuse-resistant fingerprint, or ``None`` when it exited."""\n    try:\n        raw = Path(f"/proc/{pid}/stat").read_text(encoding="utf-8")\n        close = raw.rfind(")")\n        fields = raw[close + 2 :].split()\n        argv_bytes = Path(f"/proc/{pid}/cmdline").read_bytes()\n        return {\n            "pid": pid,\n            "state": fields[0],\n            "ppid": int(fields[1]),\n            "pgid": int(fields[2]),\n            "sid": int(fields[3]),\n            "start_ticks": int(fields[19]),\n            "argv_sha256": hashlib.sha256(\n                json.dumps(\n                    [part.decode("utf-8", errors="replace") for part in argv_bytes.split(b"\\0") if part],\n                    separators=(",", ":"),\n                ).encode("utf-8")\n            ).hexdigest(),\n        }\n    except (FileNotFoundError, IndexError, OSError, ValueError):\n        return None\n\n\ndef _server_paths(working_dir: Path) -> dict[str, Path]:\n    return {\n        "identity": working_dir / IDENTITY_FILENAME,\n        "provenance": working_dir / PROVENANCE_FILENAME,\n        "log": working_dir / LOG_FILENAME,\n        "metrics": working_dir / METRICS_FILENAME,\n        "teardown": working_dir / TEARDOWN_FILENAME,\n        "tool_smoke": working_dir / TOOL_SMOKE_FILENAME,\n        "harmony_preflight": working_dir / HARMONY_PREFLIGHT_FILENAME,\n        "startup_failure": working_dir / STARTUP_FAILURE_FILENAME,\n    }\n\n\ndef _bootstrap_path(bundle_dir: Path) -> Path:\n    candidates = [bundle_dir / "serving_setup.py", Path("/tmp/arc3-gptoss-source.yNkUMhA1/serving_setup.py")]\n    for candidate in candidates:\n        if candidate.is_file() and _sha256(candidate) == BOOTSTRAP_SHA256:\n            return candidate\n    raise RuntimeError("The pinned serving_setup.py bootstrap was not available with its approved SHA-256.")\n\n\ndef _load_bootstrap(bundle_dir: Path, working_dir: Path) -> tuple[Any, Path]:\n    """Load the pinned bootstrap without executing its Qwen entry point."""\n    source = _bootstrap_path(bundle_dir)\n    saved = {name: os.environ.get(name) for name in ("TAAF_KAGGLE_BUNDLE_DIR", "TAAF_KAGGLE_WORKING_DIR", "TAAF_KAGGLE_SETUP_ENV")}\n    try:\n        os.environ["TAAF_KAGGLE_BUNDLE_DIR"] = str(bundle_dir)\n        os.environ["TAAF_KAGGLE_WORKING_DIR"] = str(working_dir)\n        os.environ["TAAF_KAGGLE_SETUP_ENV"] = str(working_dir / "gptoss-ignored-setup-env.json")\n        module_name = f"arc3_pinned_runtime_{hashlib.sha256(str(source).encode()).hexdigest()[:16]}"\n        spec = importlib.util.spec_from_file_location(module_name, source)\n        if spec is None or spec.loader is None:\n            raise RuntimeError(f"Could not load pinned bootstrap: {source}")\n        module = importlib.util.module_from_spec(spec)\n        spec.loader.exec_module(module)\n        return module, source\n    finally:\n        for name, value in saved.items():\n            if value is None:\n                os.environ.pop(name, None)\n            else:\n                os.environ[name] = value\n\n\ndef _model_candidates() -> list[Path]:\n    configured = os.environ.get("ARC3_GPTOSS_MODEL_DIR", "").strip()\n    candidates = [Path(configured)] if configured else []\n    candidates.extend(\n        [\n            Path("/kaggle/input/models/danielhanchen/gpt-oss-120b/transformers/default/1"),\n            Path("/kaggle/input/models/danielhanchen/gpt-oss-120b/Transformers/default/1"),\n        ]\n    )\n    return candidates\n\n\ndef _validate_model_dir(model_dir: Path) -> dict[str, Any]:\n    config_path = model_dir / "config.json"\n    if not config_path.is_file() or model_dir.is_symlink():\n        raise FileNotFoundError(f"GPT-OSS attachment is missing a regular config.json: {model_dir}")\n    # The Kaggle attachment is intentionally exact: .../gpt-oss-120b/Transformers/default/1.\n    parts = [part.lower() for part in model_dir.parts]\n    if len(parts) < 4 or parts[-3:] != ["transformers", "default", "1"] or "gpt-oss-120b" not in parts:\n        raise RuntimeError(f"Unexpected GPT-OSS attachment path: {model_dir}")\n    config = _json_read(config_path)\n    if config.get("model_type") != MODEL_TYPE:\n        raise RuntimeError(f"Expected model_type={MODEL_TYPE!r}, got {config.get(\'model_type\')!r}.")\n    return {"path": str(model_dir), "config_sha256": _sha256(config_path), "model_type": MODEL_TYPE}\n\n\ndef resolve_model_dir() -> tuple[Path, dict[str, Any]]:\n    errors: list[str] = []\n    for candidate in _model_candidates():\n        try:\n            return candidate, _validate_model_dir(candidate)\n        except (FileNotFoundError, RuntimeError) as exc:\n            errors.append(str(exc))\n    raise FileNotFoundError("Could not resolve the exact GPT-OSS attachment. " + " | ".join(errors))\n\n\ndef _runtime_environment(runtime_root: Path, working_dir: Path) -> tuple[dict[str, str], dict[str, Any]]:\n    """Derive paths only; do not call the Qwen-only runtime_environment helper."""\n    site = runtime_root / "usr/local/lib/python3.12/dist-packages"\n    cuda_home = runtime_root / "usr/local/cuda-13.0"\n    cuda_lib = cuda_home / "targets/x86_64-linux/lib"\n    cutlass_nested = site / "nvidia_cutlass_dsl/dsl_packages"\n    required = [site / "vllm/__init__.py", site / "torch/__init__.py", cuda_home / "bin/nvcc", cuda_lib / "libcudart.so", cutlass_nested / "cutlass/__init__.py"]\n    missing = [str(path) for path in required if not path.is_file()]\n    if missing:\n        raise RuntimeError("Verified runtime lacks GPT-OSS serving prerequisites: " + ", ".join(missing))\n\n    cache_root = working_dir / "gptoss-cache"\n    compile_root = working_dir / "gptoss-compile-cache"\n    temp_root = working_dir / "gptoss-tmp"\n    for path in (cache_root, compile_root, temp_root):\n        path.mkdir(parents=True, exist_ok=True)\n    nvidia_libs = sorted(str(path) for path in (site / "nvidia").glob("*/lib") if path.is_dir())\n    library_dirs = [str(cuda_lib), str(cuda_home / "lib64"), str(site / "torch/lib"), *nvidia_libs,\n                    "/usr/local/nvidia/lib64", "/usr/lib/x86_64-linux-gnu"]\n    env = os.environ.copy()\n    for key in list(env):\n        if key == "PYTORCH_ALLOC_CONF" or "QWEN" in key or "RADIXARK" in key or "PLE" in key:\n            env.pop(key, None)\n    # PYTHONPATH does not execute the runtime\'s .pth file that adds CUTLASS DSL.\n    env["PYTHONPATH"] = os.pathsep.join([str(cutlass_nested), str(site), env.get("PYTHONPATH", "")]).rstrip(os.pathsep)\n    env["PATH"] = os.pathsep.join([str(cuda_home / "bin"), env.get("PATH", "")]).rstrip(os.pathsep)\n    # CUDA 13\'s unversioned libcudart linker name lives in cuda_lib.  Keeping it\n    # first prevents the historical host -lcudart lookup failure.\n    env["LD_LIBRARY_PATH"] = os.pathsep.join(\n        [path for path in library_dirs if Path(path).is_dir()] + ([env["LD_LIBRARY_PATH"]] if env.get("LD_LIBRARY_PATH") else [])\n    )\n    # GCC\'s link-time search is distinct from the loader\'s LD_LIBRARY_PATH.\n    env["LIBRARY_PATH"] = os.pathsep.join(\n        [path for path in library_dirs if Path(path).is_dir()] + ([env["LIBRARY_PATH"]] if env.get("LIBRARY_PATH") else [])\n    )\n    env.update(\n        {\n            "CUDA_VISIBLE_DEVICES": "0",\n            "CUDA_DEVICE_ORDER": "PCI_BUS_ID",\n            "CUDA_HOME": str(cuda_home),\n            "CUDACXX": str(cuda_home / "bin/nvcc"),\n            "HF_HOME": str(cache_root / "huggingface"),\n            "XDG_CACHE_HOME": str(cache_root),\n            "TORCH_HOME": str(cache_root / "torch"),\n            "TORCHINDUCTOR_CACHE_DIR": str(compile_root / "torchinductor"),\n            "TRITON_CACHE_DIR": str(compile_root / "triton"),\n            "CUDA_CACHE_PATH": str(compile_root / "cuda"),\n            "TMPDIR": str(temp_root),\n            "HF_HUB_OFFLINE": "1",\n            "HF_DATASETS_OFFLINE": "1",\n            "TRANSFORMERS_OFFLINE": "1",\n            "VLLM_NO_USAGE_STATS": "1",\n            "VLLM_ENABLE_CUDA_COMPATIBILITY": "0",\n            "DO_NOT_TRACK": "1",\n            "VLLM_WORKER_MULTIPROC_METHOD": "spawn",\n            "PYTHONDONTWRITEBYTECODE": "1",\n            "TOKENIZERS_PARALLELISM": "false",\n            "LOCAL_ANALYZER_BASE_URL": BASE_URL,\n            "OPENAI_BASE_URL": BASE_URL,\n            "BASE_URL": BASE_URL,\n            "LOCAL_ANALYZER_PROVIDER": "vllm",\n            "OPENAI_PROVIDER": "vllm",\n            "LOCAL_ANALYZER_MODEL_ID": MODEL_NAME,\n            "INFERENCE_ANALYZER_MODEL": MODEL_NAME,\n            "MODEL_NAME": MODEL_NAME,\n            "LOCAL_ANALYZER_API_KEY": "offline",\n            "OPENAI_API_KEY": "offline",\n            "LOCAL_ANALYZER_CONTEXT_WINDOW": str(MAX_MODEL_LEN),\n            "LOCAL_ANALYZER_MAX_OUTPUT": "0",\n            "LOCAL_ANALYZER_TOOL_STEPS": "0",\n            "LOCAL_ANALYZER_TOOL_TIMEOUT": "30",\n            "LOCAL_ANALYZER_TOOL_OUTPUT_TOKENS": "1024",\n            "LOCAL_ANALYZER_YIELD_SECONDS": "300",\n            "LOCAL_ANALYZER_TEMPERATURE": "0.6",\n            "LOCAL_ANALYZER_TOP_P": "0.95",\n            "LOCAL_ANALYZER_ENABLE_THINKING": "true",\n            "MULTIMODAL_CONTEXT": "",\n        }\n    )\n    return env, {\n        "environment_strategy": "derived_from_verified_runtime_not_qwen_runtime_environment",\n        "site_packages": str(site),\n        "cuda_home": str(cuda_home),\n        "cuda13_libcudart_link_directory": str(cuda_lib),\n        "cuda13_libcudart": str(cuda_lib / "libcudart.so"),\n    }\n\n\ndef _duck_environment(env: dict[str, str]) -> dict[str, str]:\n    keys = (\n        "LOCAL_ANALYZER_PROVIDER", "OPENAI_PROVIDER", "LOCAL_ANALYZER_BASE_URL",\n        "OPENAI_BASE_URL", "BASE_URL", "LOCAL_ANALYZER_MODEL_ID",\n        "INFERENCE_ANALYZER_MODEL", "MODEL_NAME", "LOCAL_ANALYZER_API_KEY",\n        "OPENAI_API_KEY", "LOCAL_ANALYZER_CONTEXT_WINDOW", "LOCAL_ANALYZER_MAX_OUTPUT",\n        "LOCAL_ANALYZER_TOOL_STEPS", "LOCAL_ANALYZER_TOOL_TIMEOUT",\n        "LOCAL_ANALYZER_TOOL_OUTPUT_TOKENS", "LOCAL_ANALYZER_YIELD_SECONDS",\n        "LOCAL_ANALYZER_TEMPERATURE", "LOCAL_ANALYZER_TOP_P",\n        "LOCAL_ANALYZER_ENABLE_THINKING", "MULTIMODAL_CONTEXT",\n    )\n    return {key: env[key] for key in keys}\n\n\ndef _prepare_harmony_cache(working_dir: Path, env: dict[str, str]) -> dict[str, Any]:\n    """Verify the pinned offline vocabulary and expose only its cache directory."""\n    cache_dir = working_dir / HARMONY_CACHE_DIRNAME\n    vocabulary = cache_dir / HARMONY_TIKTOKEN_CACHE_KEY\n    if not vocabulary.is_file() or vocabulary.is_symlink():\n        raise FileNotFoundError(f"Pinned Harmony vocabulary is missing: {vocabulary}")\n    actual_bytes = vocabulary.stat().st_size\n    if actual_bytes != HARMONY_VOCAB_BYTES:\n        raise RuntimeError(\n            f"Pinned Harmony vocabulary has {actual_bytes} bytes; expected {HARMONY_VOCAB_BYTES}."\n        )\n    actual_sha256 = _sha256(vocabulary)\n    if actual_sha256 != HARMONY_VOCAB_SHA256:\n        raise RuntimeError("Pinned Harmony vocabulary SHA-256 did not match the approved offline asset.")\n    # A stale override can change the upstream URL and therefore the cache key.\n    # The pinned asset is for openai-harmony\'s unmodified official URL.\n    env.pop("TIKTOKEN_ENCODINGS_BASE", None)\n    env["TIKTOKEN_RS_CACHE_DIR"] = str(cache_dir)\n    return {\n        "cache_dir": str(cache_dir), "cache_key": HARMONY_TIKTOKEN_CACHE_KEY,\n        "bytes": actual_bytes, "sha256": actual_sha256,\n    }\n\n\n_HARMONY_PREFLIGHT_SOURCE = """\nimport json\nfrom importlib.metadata import version\nfrom openai_harmony import Conversation, HarmonyEncodingName, Message, Role, SystemContent, load_harmony_encoding\n\nencoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)\nconversation = Conversation.from_messages([\n    Message.from_role_and_content(Role.SYSTEM, SystemContent.new()),\n    Message.from_role_and_content(Role.USER, "Reply with only: ok."),\n])\ntokens = encoding.render_conversation_for_completion(conversation, Role.ASSISTANT)\nprint(json.dumps({"harmony_version": version("openai-harmony"), "token_count": len(tokens)}))\n"""\n\n\ndef _run_harmony_preflight(env: dict[str, str], output_path: Path, deadline: float) -> dict[str, Any]:\n    """Render a fixed conversation without CUDA or a vLLM server process."""\n    remaining = deadline - time.monotonic()\n    if remaining <= 0:\n        raise TimeoutError("GPT-OSS startup budget expired before Harmony CPU preflight.")\n    timeout = min(HARMONY_PREFLIGHT_MAX_SECONDS, remaining)\n    result: dict[str, Any] = {"phase": "running", "timeout_seconds": timeout}\n    _json_write(output_path, result)\n    try:\n        completed = subprocess.run(\n            [sys.executable, "-c", _HARMONY_PREFLIGHT_SOURCE], env=env,\n            capture_output=True, text=True, timeout=timeout, check=True,\n        )\n        value = json.loads(completed.stdout)\n        if not isinstance(value, dict) or not isinstance(value.get("harmony_version"), str) or not isinstance(value.get("token_count"), int):\n            raise RuntimeError("Harmony CPU preflight returned an invalid report.")\n        result.update({"phase": "passed", **value})\n        _json_write(output_path, result)\n        return result\n    except Exception as exc:\n        result.update({"phase": "failed", "error_type": type(exc).__name__, "error": str(exc)})\n        _json_write(output_path, result)\n        raise\n\n\ndef server_command(model_dir: Path) -> list[str]:\n    """The GPT-OSS command intentionally has no Qwen template/parser/MTP flags."""\n    return [\n        sys.executable, "-m", "vllm.entrypoints.cli.main", "serve", str(model_dir),\n        "--served-model-name", MODEL_NAME, "--host", HOST, "--port", str(PORT),\n        "--tensor-parallel-size", "1", "--gpu-memory-utilization", GPU_MEMORY_UTILIZATION,\n        "--max-model-len", str(MAX_MODEL_LEN), "--max-num-seqs", str(MAX_NUM_SEQS),\n        "--enforce-eager", "--enable-prefix-caching", "--enable-auto-tool-choice",\n        "--tool-call-parser", "openai", "--no-enable-log-requests",\n        "--disable-uvicorn-access-log",\n    ]\n\n\ndef _http_bytes(url: str, timeout: float, payload: dict[str, Any] | None = None) -> bytes:\n    try:\n        request = urllib.request.Request(\n            url,\n            None if payload is None else json.dumps(payload).encode("utf-8"),\n            headers={"Content-Type": "application/json", "Accept": "application/json"},\n        )\n        with urllib.request.urlopen(request, timeout=timeout) as response:\n            return response.read()\n    except urllib.error.HTTPError as exc:\n        # Preserve just enough server evidence to diagnose a rejected request.\n        # Do not retain response headers, which can carry infrastructure details.\n        body = exc.read(4096).decode("utf-8", errors="replace").strip()\n        detail = f": {body}" if body else ""\n        raise RuntimeError(f"HTTP request failed for {url}: HTTP {exc.code}{detail}") from exc\n    except (OSError, urllib.error.URLError) as exc:\n        raise RuntimeError(f"HTTP request failed for {url}: {exc}") from exc\n\n\ndef _worker_identity(record: dict[str, Any]) -> dict[str, Any]:\n    return {key: record[key] for key in ("pid", "start_ticks", "pgid", "sid", "argv_sha256")}\n\n\ndef _snapshot_owned_workers(identity: dict[str, Any]) -> list[dict[str, Any]]:\n    """Snapshot exact session members only while the root proves ownership."""\n    matching, reason = _identity_matches(identity)\n    if not matching:\n        raise RuntimeError(f"Cannot update worker identity after root change: {reason}")\n    workers = [_worker_identity(record) for record in _owned_live_records(identity)]\n    root = [worker for worker in workers if worker["pid"] == identity["pid"]]\n    if len(root) != 1 or root[0]["start_ticks"] != identity["start_ticks"]:\n        raise RuntimeError("Owned worker snapshot lost the GPT-OSS root.")\n    return sorted(workers, key=lambda worker: int(worker["pid"]))\n\n\ndef _wait_ready(identity: dict[str, Any], paths: dict[str, Path], deadline: float) -> float:\n    while time.monotonic() < deadline:\n        live = _proc_record(int(identity["pid"]))\n        if live is None or live["state"] == "Z" or int(live["start_ticks"]) != int(identity["start_ticks"]):\n            raise RuntimeError("GPT-OSS server exited before readiness.")\n        # Persist only a root-verified snapshot; a vanished/reused root cannot\n        # erase the last known-good child identities.\n        identity["workers"] = _snapshot_owned_workers(identity)\n        identity["worker_snapshot_epoch"] = time.time()\n        _json_write(paths["identity"], identity)\n        try:\n            value = json.loads(_http_bytes(f"{BASE_URL}/models", min(5.0, max(0.1, deadline - time.monotonic()))))\n        except RuntimeError:\n            time.sleep(1.0)\n            continue\n        except json.JSONDecodeError as exc:\n            raise RuntimeError("GPT-OSS /v1/models did not return JSON.") from exc\n        if not isinstance(value, dict):\n            raise RuntimeError("GPT-OSS /v1/models did not return an object.")\n        ids = [row.get("id") for row in value.get("data", []) if isinstance(row, dict)]\n        if ids == [MODEL_NAME]:\n            return time.time()\n        if ids:\n            raise RuntimeError(f"GPT-OSS endpoint served the wrong model IDs: {ids}")\n        time.sleep(1.0)\n    raise TimeoutError("Timed out waiting for GPT-OSS /v1/models readiness.")\n\n\ndef _tool_smoke(deadline: float, output_path: Path) -> dict[str, Any]:\n    """Exercise native tool parsing before the benchmark\'s first game compiles.\n\n    This is fixed data only: it never authorizes model-produced code or tools.\n    """\n    first = {\n        "model": MODEL_NAME,\n        "messages": [{"role": "user", "content": "Call submit_number with value 4. Do not answer in prose."}],\n        "tools": [{"type": "function", "function": {"name": "submit_number", "description": "Submit one integer.", "parameters": {"type": "object", "properties": {"value": {"type": "integer"}}, "required": ["value"]}}}],\n        "tool_choice": "auto", "ignore_eos": False,\n        "temperature": 0.0, "reasoning_effort": "low", "max_tokens": 512,\n    }\n    first_timeout = min(120.0, max(0.1, deadline - time.monotonic()))\n    result: dict[str, Any] = {"first_request": first}\n    _json_write(output_path, result)\n    try:\n        first_response = json.loads(_http_bytes(f"{BASE_URL}/chat/completions", first_timeout, first))\n    except Exception as exc:\n        result["first_error"] = {"type": type(exc).__name__, "message": str(exc)}\n        _json_write(output_path, result)\n        raise\n    result["first_response"] = first_response\n    _json_write(output_path, result)\n    message = ((first_response.get("choices") or [{}])[0].get("message") or {})\n    calls = message.get("tool_calls") or []\n    if len(calls) != 1 or (calls[0].get("function") or {}).get("name") != "submit_number":\n        raise RuntimeError(f"GPT-OSS native tool smoke returned no submit_number call: {first_response}")\n    call_id = calls[0].get("id")\n    if not isinstance(call_id, str) or not call_id.strip():\n        raise RuntimeError(f"GPT-OSS native tool smoke returned an invalid call ID: {calls}")\n    raw_arguments = (calls[0].get("function") or {}).get("arguments", "{}")\n    arguments = raw_arguments if isinstance(raw_arguments, dict) else json.loads(raw_arguments)\n    if arguments.get("value") != 4:\n        raise RuntimeError(f"GPT-OSS native tool smoke returned wrong arguments: {calls}")\n    assistant_message = {"role": "assistant", "content": message.get("content"), "tool_calls": calls}\n    for field in ("reasoning", "reasoning_content"):\n        if field in message:\n            assistant_message[field] = message[field]\n    second = {\n        "model": MODEL_NAME,\n        "messages": [*first["messages"], assistant_message, {"role": "tool", "tool_call_id": call_id, "content": "{\\"accepted\\":true}"}],\n        "tools": first["tools"], "tool_choice": "auto", "ignore_eos": False,\n        "temperature": 0.0, "reasoning_effort": "low", "max_tokens": 512,\n    }\n    second_timeout = min(120.0, max(0.1, deadline - time.monotonic()))\n    result["second_request"] = second\n    _json_write(output_path, result)\n    try:\n        second_response = json.loads(_http_bytes(f"{BASE_URL}/chat/completions", second_timeout, second))\n    except Exception as exc:\n        result["second_error"] = {"type": type(exc).__name__, "message": str(exc)}\n        _json_write(output_path, result)\n        raise\n    result["second_response"] = second_response\n    _json_write(output_path, result)\n    final_message = ((second_response.get("choices") or [{}])[0].get("message") or {})\n    if final_message.get("tool_calls"):\n        raise RuntimeError(f"GPT-OSS tool-result smoke returned unexpected tool calls: {second_response}")\n    if not isinstance(final_message.get("content"), str) or not final_message["content"].strip():\n        raise RuntimeError(f"GPT-OSS tool-result smoke returned no final response: {second_response}")\n    return result\n\n\ndef _register_atexit_guard(working_dir: Path) -> None:\n    key = str(working_dir.resolve())\n    if key in _ATEXIT_GUARDS:\n        return\n    _ATEXIT_GUARDS.add(key)\n    import atexit\n\n    def cleanup() -> None:\n        try:\n            stop_server(working_dir)\n        except Exception:\n            pass\n\n    atexit.register(cleanup)\n\n\ndef arm_shutdown_watchdog(working_dir: Path, deadline_epoch: float) -> dict[str, Any]:\n    """Arm one identity-bound, in-process deadline guard for a short smoke.\n\n    The guard checks that the same PID/start-ticks identity still owns the\n    working directory before it calls ``stop_server``.  It is deliberately\n    bounded to twenty minutes and is cancelled implicitly by successful stop.\n    """\n    if not isinstance(deadline_epoch, (int, float)):\n        raise TypeError("deadline_epoch must be an epoch timestamp.")\n    paths = _server_paths(Path(working_dir))\n    identity = _json_read(paths["identity"])\n    delay = float(deadline_epoch) - time.time()\n    if not 0 < delay <= MAX_WATCHDOG_SECONDS:\n        raise ValueError(f"watchdog deadline must be within {MAX_WATCHDOG_SECONDS}s.")\n    expected = (identity.get("pid"), identity.get("start_ticks"))\n\n    def guard() -> None:\n        time.sleep(delay)\n        try:\n            current = _json_read(paths["identity"])\n            if (current.get("pid"), current.get("start_ticks")) == expected:\n                stop_server(Path(working_dir))\n        except Exception:\n            pass\n\n    thread = threading.Thread(target=guard, name="gptoss-shutdown-guard", daemon=True)\n    thread.start()\n    return {"armed": True, "deadline_epoch": float(deadline_epoch), "pid": expected[0], "start_ticks": expected[1]}\n\n\ndef _identity_matches(identity: dict[str, Any]) -> tuple[bool, str | None]:\n    try:\n        pid = int(identity["pid"])\n        ticks = int(identity["start_ticks"])\n        pgid = int(identity["pgid"])\n        sid = int(identity["sid"])\n    except (KeyError, TypeError, ValueError):\n        return False, "invalid_identity_numbers"\n    if identity.get("boot_id") != _boot_id():\n        return False, "boot_id_changed"\n    current = _proc_record(pid)\n    if current is None or current["state"] == "Z":\n        return False, "root_not_live"\n    if ticks != current["start_ticks"]:\n        return False, "pid_identity_changed"\n    if pgid != pid or sid != pid or current["pgid"] != pgid or current["sid"] != sid:\n        return False, "process_group_not_owned"\n    if identity.get("argv_sha256") != current["argv_sha256"]:\n        return False, "argv_identity_changed"\n    return True, None\n\n\ndef _alive_identity(identity: dict[str, Any]) -> bool:\n    current = _proc_record(int(identity["pid"]))\n    return current is not None and current["state"] != "Z" and int(current["start_ticks"]) == int(identity["start_ticks"])\n\n\ndef _owned_live_records(identity: dict[str, Any]) -> list[dict[str, Any]]:\n    """Enumerate the isolated session while it is still bound to our launch."""\n    try:\n        pgid, sid = int(identity["pgid"]), int(identity["sid"])\n    except (KeyError, TypeError, ValueError):\n        return []\n    records: list[dict[str, Any]] = []\n    try:\n        proc_entries = list(Path("/proc").iterdir())\n    except OSError:\n        return []\n    for entry in proc_entries:\n        if not entry.name.isdigit():\n            continue\n        record = _proc_record(int(entry.name))\n        if record is not None and record["state"] != "Z" and record["pgid"] == pgid and record["sid"] == sid:\n            records.append(record)\n    return records\n\n\ndef _same_live_record(expected: dict[str, Any]) -> dict[str, Any] | None:\n    current = _proc_record(int(expected["pid"]))\n    if current is None or current["state"] == "Z":\n        return None\n    for key in ("start_ticks", "pgid", "sid", "argv_sha256"):\n        if current.get(key) != expected.get(key):\n            return None\n    return current\n\n\ndef _saved_child_workers(identity: dict[str, Any]) -> tuple[list[dict[str, Any]], list[str]]:\n    """Validate persisted workers without inferring ownership from a name."""\n    workers = identity.get("workers")\n    required = {"pid", "start_ticks", "pgid", "sid", "argv_sha256"}\n    errors: list[str] = []\n    if not isinstance(workers, list):\n        return [], ["saved_workers_missing"]\n    valid: list[dict[str, Any]] = []\n    for worker in workers:\n        if not isinstance(worker, dict) or set(worker) != required:\n            errors.append("saved_worker_schema_invalid")\n            continue\n        if not all(isinstance(worker[key], int) for key in ("pid", "start_ticks", "pgid", "sid")) or not isinstance(worker["argv_sha256"], str):\n            errors.append("saved_worker_fields_invalid")\n            continue\n        valid.append(worker)\n    roots = [worker for worker in valid if worker["pid"] == identity.get("pid")]\n    if len(roots) != 1 or any(roots[0].get(key) != identity.get(key) for key in ("start_ticks", "pgid", "sid", "argv_sha256")):\n        errors.append("saved_root_worker_mismatch")\n    return [worker for worker in valid if worker["pid"] != identity.get("pid")], errors\n\n\ndef _terminate_saved_children(workers: list[dict[str, Any]]) -> tuple[list[dict[str, Any]], list[str]]:\n    """Bounded exact-PID teardown for a session whose root already exited."""\n    errors: list[str] = []\n    live: list[dict[str, Any]] = []\n    for worker in workers:\n        current = _same_live_record(worker)\n        if current is not None:\n            live.append(worker)\n        elif _proc_record(int(worker["pid"])) is not None:\n            errors.append(f"saved_child_identity_changed:{worker[\'pid\']}")\n    for worker in live:\n        try:\n            os.kill(int(worker["pid"]), signal.SIGTERM)\n        except ProcessLookupError:\n            pass\n    deadline = time.monotonic() + TERM_GRACE_SECONDS\n    while any(_same_live_record(worker) is not None for worker in live) and time.monotonic() < deadline:\n        time.sleep(0.2)\n    survivors = [worker for worker in live if _same_live_record(worker) is not None]\n    for worker in survivors:\n        # Recheck immediately before every exact PID SIGKILL.\n        if _same_live_record(worker) is None:\n            continue\n        try:\n            os.kill(int(worker["pid"]), signal.SIGKILL)\n        except ProcessLookupError:\n            pass\n    deadline = time.monotonic() + KILL_GRACE_SECONDS\n    while any(_same_live_record(worker) is not None for worker in survivors) and time.monotonic() < deadline:\n        time.sleep(0.2)\n    return [worker for worker in survivors if _same_live_record(worker) is not None], errors\n\n\ndef _gpu_rows() -> tuple[list[dict[str, Any]], bool]:\n    try:\n        completed = subprocess.run(\n            ["nvidia-smi", "--query-compute-apps=pid,process_name,used_memory", "--format=csv,noheader,nounits"],\n            capture_output=True, text=True, timeout=10.0, check=False,\n        )\n    except (OSError, subprocess.TimeoutExpired) as exc:\n        return [{"query_error": str(exc)}], False\n    if completed.returncode != 0:\n        return [{"query_error": completed.stderr.strip() or "nvidia-smi failed"}], False\n    rows: list[dict[str, Any]] = []\n    for line in completed.stdout.splitlines():\n        parts = [part.strip() for part in line.split(",")]\n        if len(parts) < 3:\n            continue\n        try:\n            pid = int(parts[0])\n        except ValueError:\n            continue\n        record = _proc_record(pid)\n        rows.append({"pid": pid, "process_name": parts[1], "used_memory_mib": parts[2], "start_ticks": None if record is None else record["start_ticks"]})\n    return rows, True\n\n\ndef gpu_rows() -> list[dict[str, Any]]:\n    """Return a fresh GPU process snapshot; query failures are explicit rows."""\n    return _gpu_rows()[0]\n\n\ndef _capture_metrics(paths: dict[str, Path], errors: list[str]) -> bool:\n    if paths["metrics"].is_file() and paths["metrics"].stat().st_size > 0:\n        return True\n    try:\n        metrics = _http_bytes(f"http://{HOST}:{PORT}/metrics", METRICS_TIMEOUT_SECONDS)\n        if not metrics:\n            raise RuntimeError("metrics endpoint returned an empty response")\n        paths["metrics"].write_bytes(metrics)\n        return True\n    except Exception as exc:  # stop must still terminate an owned server.\n        errors.append(f"metrics_capture:{type(exc).__name__}:{exc}")\n        return False\n\n\ndef start_server(bundle_dir: Path, working_dir: Path, timeout_seconds: int = 900) -> dict[str, Any]:\n    """Extract, launch, and return only after the offline endpoint is ready."""\n    if type(timeout_seconds) is not int or not 1 <= timeout_seconds <= 1800:\n        raise ValueError("timeout_seconds must be an integer from 1 through 1800.")\n    bundle_dir, working_dir = Path(bundle_dir), Path(working_dir)\n    working_dir.mkdir(parents=True, exist_ok=True)\n    paths = _server_paths(working_dir)\n    prior = stop_server(working_dir)\n    if paths["identity"].is_file() and not prior.get("shutdown_ok", False):\n        raise RuntimeError(f"A prior owned GPT-OSS server did not shut down: {prior}")\n    try:\n        with socket.create_connection((HOST, PORT), timeout=0.25):\n            raise RuntimeError(f"{HOST}:{PORT} is already occupied by an unowned process.")\n    except OSError:\n        pass\n\n    deadline = time.monotonic() + timeout_seconds\n    identity: dict[str, Any] | None = None\n    process: subprocess.Popen[str] | None = None\n    provenance: dict[str, Any] | None = None\n    try:\n        setup, source = _load_bootstrap(bundle_dir, working_dir)\n        if getattr(setup, "VLLM_RUNTIME_MANIFEST_SHA256", None) != BOOTSTRAP_RUNTIME_MANIFEST_SHA256:\n            raise RuntimeError("Pinned bootstrap has an unexpected runtime manifest pin.")\n        if getattr(setup, "VLLM_VERSION", None) != BOOTSTRAP_VLLM_VERSION:\n            raise RuntimeError("Pinned bootstrap has an unexpected vLLM version pin.")\n        model_dir, model_provenance = resolve_model_dir()\n        runtime_dir = setup.resolve_runtime_dir()\n        verification = setup.verify_and_extract_runtime(runtime_dir, full_layer_hashes=True)\n        if time.monotonic() >= deadline:\n            raise TimeoutError("GPT-OSS startup budget expired during runtime extraction.")\n        env, environment_provenance = _runtime_environment(Path(setup.RUNTIME_ROOT), working_dir)\n        provenance = {\n            "schema_version": 1, "phase": "preflight", "runtime_dataset": RUNTIME_DATASET,\n            "bootstrap_path": str(source), "bootstrap_sha256": BOOTSTRAP_SHA256,\n            "bootstrap_runtime_manifest_sha256": BOOTSTRAP_RUNTIME_MANIFEST_SHA256,\n            "bootstrap_vllm_version": BOOTSTRAP_VLLM_VERSION, "runtime_verification": verification,\n            "model": model_provenance, "environment": environment_provenance,\n        }\n        _json_write(paths["provenance"], provenance)\n        provenance["harmony_cache"] = _prepare_harmony_cache(working_dir, env)\n        _json_write(paths["provenance"], provenance)\n        provenance["harmony_preflight"] = _run_harmony_preflight(env, paths["harmony_preflight"], deadline)\n        provenance["phase"] = "preflight_complete"\n        _json_write(paths["provenance"], provenance)\n        command = server_command(model_dir)\n        provenance.update({"phase": "launching", "actual_command": command})\n        _json_write(paths["provenance"], provenance)\n        paths["log"].unlink(missing_ok=True)\n        with paths["log"].open("w", encoding="utf-8") as log_handle:\n            process = subprocess.Popen(command, env=env, stdout=log_handle, stderr=subprocess.STDOUT, text=True, start_new_session=True)\n        root = _proc_record(process.pid)\n        if root is None or root["pgid"] != process.pid or root["sid"] != process.pid:\n            raise RuntimeError(f"Launched GPT-OSS root lacks an isolated session: {root}")\n        argv_sha256 = hashlib.sha256(json.dumps(command, separators=(",", ":")).encode("utf-8")).hexdigest()\n        if root["argv_sha256"] != argv_sha256:\n            raise RuntimeError("Launched GPT-OSS argv did not match the recorded command.")\n        identity = {\n            "schema_version": 1, "backend": "vllm", "model_name": MODEL_NAME,\n            "boot_id": _boot_id(), "host": HOST, "port": PORT, "pid": process.pid,\n            "start_ticks": root["start_ticks"], "pgid": root["pgid"], "sid": root["sid"],\n            "argv": command, "argv_sha256": argv_sha256, "started_epoch": time.time(),\n            "phase": "starting", "server_ready_epoch": None,\n            "workers": [_worker_identity(root)], "worker_snapshot_epoch": time.time(),\n        }\n        _json_write(paths["identity"], identity)\n        _wait_ready(identity, paths, deadline)\n        if time.monotonic() >= deadline:\n            raise TimeoutError("GPT-OSS startup budget expired before tool readiness smoke.")\n        _tool_smoke(deadline, paths["tool_smoke"])\n        if time.monotonic() >= deadline:\n            raise TimeoutError("GPT-OSS startup budget expired during tool readiness smoke.")\n        identity["workers"] = _snapshot_owned_workers(identity)\n        identity["worker_snapshot_epoch"] = time.time()\n        ready_epoch = time.time()\n        identity["phase"] = "ready"\n        identity["server_ready_epoch"] = ready_epoch\n        _json_write(paths["identity"], identity)\n        _register_atexit_guard(working_dir)\n        provenance.update({\n            "phase": "ready", "actual_command": command, "tool_smoke_path": str(paths["tool_smoke"]),\n            "tool_smoke_sha256": _sha256(paths["tool_smoke"]), "server_ready_epoch": ready_epoch,\n        })\n        _json_write(paths["provenance"], provenance)\n        return {"environment": _duck_environment(env), "server_ready_epoch": ready_epoch, "provenance": provenance}\n    except Exception as exc:\n        teardown: dict[str, Any]\n        if identity is not None:\n            try:\n                teardown = stop_server(working_dir)\n            except Exception as teardown_exc:\n                teardown = {"attempted": True, "error_type": type(teardown_exc).__name__, "error": str(teardown_exc)}\n        elif process is not None and process.poll() is None:\n            # Popen is our exact child even if it failed before identity capture.\n            try:\n                process.terminate()\n                try:\n                    process.wait(timeout=5)\n                except subprocess.TimeoutExpired:\n                    process.kill()\n                    process.wait(timeout=5)\n                teardown = {"attempted": True, "method": "direct_child", "returncode": process.returncode}\n            except Exception as teardown_exc:\n                teardown = {"attempted": True, "method": "direct_child", "error_type": type(teardown_exc).__name__, "error": str(teardown_exc)}\n        else:\n            teardown = {"attempted": False, "reason": "process_not_started"}\n        if provenance is not None:\n            provenance.update({"phase": "failed", "failure": {"type": type(exc).__name__, "message": str(exc)}})\n            _json_write(paths["provenance"], provenance)\n        _json_write(paths["startup_failure"], {\n            "exception_type": type(exc).__name__, "exception_message": str(exc), "teardown": teardown,\n        })\n        raise\n\n\ndef stop_server(working_dir: Path) -> dict[str, Any]:\n    """Capture metrics then terminate only the PID-reuse-checked owned session."""\n    working_dir = Path(working_dir)\n    paths = _server_paths(working_dir)\n    previous = _json_read(paths["teardown"]) if paths["teardown"].is_file() else None\n    repeated_success = previous is not None and previous.get("shutdown_ok") is True\n    errors: list[str] = []\n    if not paths["identity"].is_file():\n        rows, query_ok = _gpu_rows()\n        result = {"shutdown_ok": False, "metrics_preserved": paths["metrics"].is_file(), "errors": ["identity_missing"], "post_gpu_rows": rows, "gpu_query_ok": query_ok}\n        if previous is None:\n            _json_write(paths["teardown"], result)\n        return result\n    identity = _json_read(paths["identity"])\n    matching, reason = _identity_matches(identity)\n    root_dead = reason == "root_not_live"\n    saved_children, saved_worker_errors = _saved_child_workers(identity)\n    owned_at_start_of_stop = matching or (root_dead and not saved_worker_errors)\n    metrics_preserved = _capture_metrics(paths, errors) if matching else paths["metrics"].is_file() and paths["metrics"].stat().st_size > 0\n    saved_child_survivors: list[dict[str, Any]] = []\n    if matching:\n        owned_before = _owned_live_records(identity)\n        try:\n            os.killpg(int(identity["pgid"]), signal.SIGTERM)\n        except ProcessLookupError:\n            pass\n        term_deadline = time.monotonic() + TERM_GRACE_SECONDS\n        while any(_same_live_record(record) is not None for record in owned_before) and time.monotonic() < term_deadline:\n            time.sleep(0.2)\n        survivors = [record for record in owned_before if _same_live_record(record) is not None]\n        if survivors:\n            # Revalidate immediately before SIGKILL; never kill a reused PID/group.\n            matching, reason = _identity_matches(identity)\n            if matching:\n                try:\n                    os.killpg(int(identity["pgid"]), signal.SIGKILL)\n                except ProcessLookupError:\n                    pass\n            else:\n                # The root can exit first.  Kill only the start-tick/argv matched\n                # child PIDs captured while the owned session was live.\n                for record in survivors:\n                    if _same_live_record(record) is None:\n                        continue\n                    try:\n                        os.kill(int(record["pid"]), signal.SIGKILL)\n                    except ProcessLookupError:\n                        pass\n            kill_deadline = time.monotonic() + KILL_GRACE_SECONDS\n            while any(_same_live_record(record) is not None for record in survivors) and time.monotonic() < kill_deadline:\n                time.sleep(0.2)\n    elif root_dead and not saved_worker_errors:\n        saved_child_survivors, child_errors = _terminate_saved_children(saved_children)\n        errors.extend(child_errors)\n    else:\n        if not repeated_success:\n            errors.append(f"identity_rejected:{reason}")\n        errors.extend(saved_worker_errors)\n    rows, query_ok = _gpu_rows()\n    owned_after = [row for row in rows if row.get("pid") == identity.get("pid") and row.get("start_ticks") == identity.get("start_ticks")]\n    cpu_survivors = _owned_live_records(identity) if matching else saved_child_survivors\n    shutdown_ok = (owned_at_start_of_stop or repeated_success) and not errors and not cpu_survivors and query_ok and not owned_after and metrics_preserved\n    result = {"shutdown_ok": shutdown_ok, "metrics_preserved": metrics_preserved, "errors": errors, "post_gpu_rows": rows, "gpu_query_ok": query_ok, "owned_gpu_rows_after": owned_after, "cpu_survivors": [{"pid": row["pid"], "start_ticks": row["start_ticks"]} for row in cpu_survivors], "pid": identity.get("pid"), "start_ticks": identity.get("start_ticks")}\n    if repeated_success:\n        # Do not overwrite the first successful metrics/lifecycle evidence.\n        result["first_success_preserved"] = True\n        result["repeated"] = True\n        return result\n    _json_write(paths["teardown"], result)\n    return result\n')
sys.path.insert(0, str(_gptoss_module_dir))
import gptoss_runtime
_gptoss_start = gptoss_runtime.start_server(BUNDLE_DIR, WORKING_DIR, timeout_seconds=900)
os.environ.update(_gptoss_start["environment"])
GATE1_SERVER_READY_EPOCH = float(_gptoss_start["server_ready_epoch"])
GATE1_SERVER_STARTUP_SECONDS = GATE1_SERVER_READY_EPOCH - NOTEBOOK_START_EPOCH
# In-process cleanup for later cell failures; Kaggle's 1200s run limit is the
# external bound if the notebook kernel itself disappears.
_gptoss_watchdog = gptoss_runtime.arm_shutdown_watchdog(
    WORKING_DIR, GATE1_SERVER_READY_EPOCH + 300.0)
print("GATE1_SERVER_READY " + json.dumps({
    "epoch": GATE1_SERVER_READY_EPOCH,
    "startup_seconds": GATE1_SERVER_STARTUP_SECONDS,
    "model": os.environ["LOCAL_ANALYZER_MODEL_ID"],
}), flush=True)


In [ ]:
# Restore the deployment target and record the real submission state on it.
with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION

# Restore the benchmark and point its outputs at the Kaggle working dir.
with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

In [ ]:
# Exact tested memory adapter embedded for offline Kaggle execution.
from pathlib import Path as _memory_Path
import sys as _memory_sys
_memory_package = WORKING_DIR / 'arc3'
_memory_package.mkdir(parents=True, exist_ok=True)
(_memory_package / '__init__.py').write_text('')
(_memory_package / 'effects.py').write_text('"""Effect ledger: what each action actually did, per state and overall.\n\nTwo jobs. Per state, it stops the agent re-probing an action that did nothing here\nbefore -- under a squared metric a repeated no-op is pure loss. Globally, it learns\nwhich actions matter in *this* game, so exploration starts with the ones that have\npaid off rather than cycling the full action space.\n"""\n\nfrom __future__ import annotations\n\nfrom collections import defaultdict\nfrom dataclasses import dataclass, field\n\nActionKey = tuple[int, int | None, int | None]  # (action id, x, y) -- x/y for ACTION6\n\n\n@dataclass\nclass ActionStats:\n    attempts: int = 0\n    changes: int = 0  # times the board actually moved\n    level_advances: int = 0\n    cells_changed: int = 0\n\n    @property\n    def change_rate(self) -> float:\n        return self.changes / self.attempts if self.attempts else 0.0\n\n    @property\n    def mean_cells_changed(self) -> float:\n        return self.cells_changed / self.changes if self.changes else 0.0\n\n\n@dataclass\nclass EffectLedger:\n    """Per-game record of action outcomes."""\n\n    by_action: dict[ActionKey, ActionStats] = field(\n        default_factory=lambda: defaultdict(ActionStats)\n    )\n    # (state hash, action) -> did anything change last time we tried it here\n    by_state: dict[tuple[str, ActionKey], bool] = field(default_factory=dict)\n\n    def record(\n        self,\n        state_hash: str,\n        action: ActionKey,\n        changed_cells: int,\n        level_advanced: bool,\n    ) -> None:\n        stats = self.by_action[action]\n        stats.attempts += 1\n        if changed_cells > 0:\n            stats.changes += 1\n            stats.cells_changed += changed_cells\n        if level_advanced:\n            stats.level_advances += 1\n        self.by_state[(state_hash, action)] = changed_cells > 0\n\n    def is_known_noop(self, state_hash: str, action: ActionKey) -> bool:\n        """True if this action did nothing the last time it was tried here."""\n        return self.by_state.get((state_hash, action)) is False\n\n    def tried_here(self, state_hash: str, action: ActionKey) -> bool:\n        return (state_hash, action) in self.by_state\n\n    def attempts(self, action: ActionKey) -> int:\n        stats = self.by_action.get(action)\n        return stats.attempts if stats else 0\n\n    def exploration_rank(self, action: ActionKey) -> tuple[int, float]:\n        """Sort key for choosing among actions untried in the current state.\n\n        Breadth first, then what has worked. Ranking purely by past success\n        collapses the agent onto whichever action moved the board most often,\n        which it then repeats forever -- it stops covering the action space and\n        the run stalls. Fewest global attempts wins, with effectiveness only\n        breaking ties, so all actions keep getting exercised.\n\n        Returned for use with ``min``; lower is better.\n        """\n        return (self.attempts(action), -self.value(action))\n\n    def value(self, action: ActionKey) -> float:\n        """How useful this action has proven, ignoring novelty."""\n        stats = self.by_action.get(action)\n        if stats is None or stats.attempts == 0:\n            return 1.0  # unknown actions carry the most information\n        if stats.level_advances:\n            return 0.9\n        return stats.change_rate * 0.8\n\n    def useful_colours(self) -> set[int]:\n        """Placeholder for colour bias; filled in by the agent that owns it."""\n        return set()\n\n    def reset_level(self) -> None:\n        """Per-state facts do not survive a level; per-action tendencies do."""\n        self.by_state.clear()\n')
(_memory_package / 'memory.py').write_text('"""Structured per-game memory the model commits explicitly.\n\nDuck keeps 30 assistant turns and clears six summary fields at a level\ntransition. The defect is not the clearing: durable summaries update only from\nassistant `content`, and the model reasons then emits a tool call, so the channel\nDuck reads is frequently empty. Preserving those summaries preserves nothing.\n**The model has to commit memory explicitly**, as a typed argument beside its\naction.\n\nEvery rule below exists because it is a way to silently destroy knowledge.\n\n**All-or-nothing.** The WHOLE update is validated -- every field, to its leaves --\nbefore a single byte is written. A half-applied update leaves memory in a state\nno one designed.\n\n**Omission keeps, empty never clears.** A field not mentioned retains its value;\n`[]` means "nothing to add", not "forget". A model truncated under length\npressure must not wipe the store.\n\n**Conflict is recorded, not resolved.** Status is not a precedence ladder. A\nREFUTED claim re-proposed as a guess is blocked; a CONFIRMED claim later refuted\nbecomes CONTESTED with both evidence sets intact. An earlier version let a guess\nresurrect a disproven claim and made a wrong confirmation permanently\nuncorrectable -- opposite errors from the same mistake of ordering statuses.\n\n**Evidence must exist.** A fact citing transition 999999 of a 300-transition\nledger is an assertion wearing a citation. Pass `known_transitions` and it is\nchecked.\n\n**A winning path is not a list of actions.** Replaying one into a level it was\nnot learned in, from a state it does not fit, is worse than not having it.\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport os\nfrom dataclasses import dataclass, field, replace\nfrom enum import Enum\nfrom pathlib import Path\nfrom typing import Any, Iterable\n\nfrom arc3.effects import ActionKey\n\n\nclass Status(str, Enum):\n    CONFIRMED = "CONFIRMED"  # checked against recorded transitions\n    ASSUMED = "ASSUMED"      # plausible, untested\n    REFUTED = "REFUTED"      # contradicted; kept so it is not re-proposed\n    CONTESTED = "CONTESTED"  # confirmed AND refuted; needs a discriminating probe\n\n\nEVIDENCE_REQUIRED = (Status.CONFIRMED, Status.REFUTED, Status.CONTESTED)\n\n\n@dataclass(frozen=True)\nclass Fact:\n    statement: str\n    status: Status = Status.ASSUMED\n    evidence: tuple[int, ...] = ()  # transition indices in this game\'s ledger\n\n    def __post_init__(self) -> None:\n        if not isinstance(self.statement, str) or not self.statement.strip():\n            raise ValueError("a fact with no statement carries no information")\n        if not isinstance(self.status, Status):\n            raise ValueError(f"status must be a Status, got {self.status!r}")\n        if any(not isinstance(i, int) or isinstance(i, bool) for i in self.evidence):\n            raise ValueError(f"evidence must be transition indices: {self.evidence!r}")\n        if self.status in EVIDENCE_REQUIRED and not self.evidence:\n            raise ValueError(\n                f"{self.status.value} without evidence: {self.statement!r}. "\n                "The store cannot tell knowledge from assertion otherwise"\n            )\n\n\n@dataclass(frozen=True)\nclass WinningPath:\n    """A sequence that worked, with everything needed to know when it applies.\n\n    Carrying bare actions across a level transition is how a \'memory\' feature\n    makes an agent worse: the board may have been resized, the objects renamed,\n    the mechanic changed. A path is replayable only from a matching level and\n    start signature, and only while each step\'s expectation holds.\n    """\n\n    level: int\n    start_signature: str  # raw board hash the path was learned from\n    actions: tuple[ActionKey, ...]\n    expected_outcomes: tuple[str, ...] = ()  # after-hash expected at each step\n    evidence: tuple[int, ...] = ()\n\n    def __post_init__(self) -> None:\n        if not self.actions:\n            raise ValueError("a winning path with no actions is not a path")\n        for action in self.actions:\n            _check_action(action)\n        if not self.start_signature:\n            raise ValueError("a path without a start signature cannot be matched")\n        if self.expected_outcomes and len(self.expected_outcomes) != len(self.actions):\n            raise ValueError("expected_outcomes must align 1:1 with actions")\n\n    def usable_from(self, level: int, signature: str) -> bool:\n        """Never replay unless the precondition matches exactly."""\n        return level == self.level and signature == self.start_signature\n\n\nDURABLE = ("action_semantics", "mechanics", "goal_evidence", "counterexamples",\n           "winning_paths")\nVOLATILE = ("current_plan", "level_coordinates", "goal_guesses", "object_identities")\nFACT_LISTS = ("mechanics", "goal_evidence", "counterexamples", "goal_guesses")\n\n\n@dataclass\nclass MemoryUpdate:\n    """What the model commits alongside its action. ``None`` means "not touching"."""\n\n    action_semantics: dict[ActionKey, list[Fact]] | None = None\n    mechanics: list[Fact] | None = None\n    goal_evidence: list[Fact] | None = None\n    counterexamples: list[Fact] | None = None\n    winning_paths: list[WinningPath] | None = None\n    current_plan: list[ActionKey] | None = None\n    level_coordinates: dict[str, Any] | None = None\n    goal_guesses: list[Fact] | None = None\n    object_identities: dict[str, Any] | None = None\n\n    def touched(self) -> list[str]:\n        return [n for n in DURABLE + VOLATILE if getattr(self, n) is not None]\n\n\ndef _check_action(action: Any) -> None:\n    if (not isinstance(action, tuple) or len(action) != 3\n            or not isinstance(action[0], int) or isinstance(action[0], bool)\n            or any(not (v is None or (isinstance(v, int) and not isinstance(v, bool)))\n                   for v in action[1:])):\n        raise ValueError(f"not an action key (id, x, y): {action!r}")\n\n\ndef _check_jsonable(name: str, value: Any) -> None:\n    try:\n        json.dumps(value, default=_json_default)\n    except (TypeError, ValueError) as exc:\n        raise ValueError(f"{name} must be JSON-serialisable: {exc}") from exc\n\n\ndef _json_default(obj: Any) -> Any:\n    if isinstance(obj, tuple):\n        return list(obj)\n    raise TypeError(f"{type(obj).__name__} is not serialisable")\n\n\ndef validate(update: MemoryUpdate, known_transitions: set[int] | None = None) -> None:\n    """Validate EVERY field to its leaves. Raises before anything is written."""\n    for name in FACT_LISTS:\n        value = getattr(update, name)\n        if value is None:\n            continue\n        if not isinstance(value, list) or any(not isinstance(f, Fact) for f in value):\n            raise ValueError(f"{name} must be a list of Fact, got {value!r}")\n        _check_evidence(name, value, known_transitions)\n\n    if update.action_semantics is not None:\n        if not isinstance(update.action_semantics, dict):\n            raise ValueError("action_semantics must be a dict keyed by action")\n        for key, facts in update.action_semantics.items():\n            _check_action(key)\n            if not isinstance(facts, list) or any(not isinstance(f, Fact) for f in facts):\n                raise ValueError(f"action_semantics[{key}] must be a list of Fact")\n            _check_evidence("action_semantics", facts, known_transitions)\n\n    if update.winning_paths is not None:\n        if not isinstance(update.winning_paths, list) or any(\n                not isinstance(p, WinningPath) for p in update.winning_paths):\n            raise ValueError("winning_paths must be a list of WinningPath")\n        if known_transitions is not None:\n            for path in update.winning_paths:\n                _check_indices("winning_paths", path.evidence, known_transitions)\n\n    if update.current_plan is not None:\n        if not isinstance(update.current_plan, list):\n            raise ValueError(f"current_plan must be a list, got {update.current_plan!r}")\n        for action in update.current_plan:\n            _check_action(action)\n\n    for name in ("level_coordinates", "object_identities"):\n        value = getattr(update, name)\n        if value is None:\n            continue\n        if not isinstance(value, dict):\n            raise ValueError(f"{name} must be a dict, got {value!r}")\n        _check_jsonable(name, value)\n\n\ndef _check_evidence(name: str, facts: Iterable[Fact],\n                    known: set[int] | None) -> None:\n    if known is None:\n        return\n    for fact in facts:\n        _check_indices(f"{name}: {fact.statement!r}", fact.evidence, known)\n\n\ndef _check_indices(label: str, indices: Iterable[int], known: set[int]) -> None:\n    missing = [i for i in indices if i not in known]\n    if missing:\n        raise ValueError(\n            f"{label} cites transitions that do not exist: {missing}. "\n            "A citation to nothing is an assertion wearing a citation"\n        )\n\n\ndef _revise(incoming: Fact, existing: Fact) -> Fact | None:\n    """Belief revision for one statement. None means the incoming fact is blocked.\n\n    Not a precedence ladder:\n\n    ```\n    existing    incoming     result\n    ASSUMED     CONFIRMED    upgrade, merge evidence\n    ASSUMED     REFUTED      refute, merge evidence\n    CONFIRMED   ASSUMED      blocked -- a guess cannot unseat a verified claim\n    CONFIRMED   REFUTED      CONTESTED, both evidence sets kept\n    REFUTED     ASSUMED      blocked -- a disproven claim is not re-proposed\n    REFUTED     CONFIRMED    CONTESTED, both evidence sets kept\n    CONTESTED   anything     stays CONTESTED, evidence accumulates\n    ```\n    """\n    merged = tuple(dict.fromkeys(existing.evidence + incoming.evidence))\n\n    if existing.status is Status.CONTESTED:\n        return replace(existing, evidence=merged)\n    if incoming.status is Status.ASSUMED and existing.status is not Status.ASSUMED:\n        return None\n    if existing.status is Status.ASSUMED:\n        return replace(incoming, evidence=merged)\n    if existing.status == incoming.status:\n        return replace(existing, evidence=merged)\n    # CONFIRMED vs REFUTED, in either direction.\n    return replace(existing, status=Status.CONTESTED, evidence=merged)\n\n\ndef _merge_facts(existing: list[Fact], incoming: Iterable[Fact]\n                 ) -> tuple[list[Fact], list[str]]:\n    """Returns (merged, blocked statements)."""\n    out = list(existing)\n    index = {f.statement: i for i, f in enumerate(out)}\n    blocked: list[str] = []\n    for fact in incoming:\n        position = index.get(fact.statement)\n        if position is None:\n            index[fact.statement] = len(out)\n            out.append(fact)\n            continue\n        revised = _revise(fact, out[position])\n        if revised is None:\n            blocked.append(fact.statement)\n        else:\n            out[position] = revised\n    return out, blocked\n\n\n@dataclass\nclass MemoryDelta:\n    changed: list[str] = field(default_factory=list)\n    blocked: list[str] = field(default_factory=list)  # re-proposed refuted claims\n    contested: list[str] = field(default_factory=list)\n\n\n@dataclass\nclass GameMemory:\n    """Everything known about ONE game. Never shared across games."""\n\n    game_id: str\n    level: int = 1\n    action_semantics: dict[ActionKey, list[Fact]] = field(default_factory=dict)\n    mechanics: list[Fact] = field(default_factory=list)\n    goal_evidence: list[Fact] = field(default_factory=list)\n    counterexamples: list[Fact] = field(default_factory=list)\n    winning_paths: list[WinningPath] = field(default_factory=list)\n    current_plan: list[ActionKey] = field(default_factory=list)\n    level_coordinates: dict[str, Any] = field(default_factory=dict)\n    goal_guesses: list[Fact] = field(default_factory=list)\n    object_identities: dict[str, Any] = field(default_factory=dict)\n    unpersisted: bool = False\n\n    def snapshot(self) -> dict[str, Any]:\n        """JSON-safe and REVERSIBLE -- action keys stay lists, not strings."""\n        return {\n            "game_id": self.game_id,\n            "level": self.level,\n            "action_semantics": [[list(k), [_fact(f) for f in v]]\n                                 for k, v in self.action_semantics.items()],\n            "mechanics": [_fact(f) for f in self.mechanics],\n            "goal_evidence": [_fact(f) for f in self.goal_evidence],\n            "counterexamples": [_fact(f) for f in self.counterexamples],\n            "winning_paths": [_path(p) for p in self.winning_paths],\n            "current_plan": [list(a) for a in self.current_plan],\n            "level_coordinates": json.loads(json.dumps(self.level_coordinates,\n                                                       default=_json_default)),\n            "goal_guesses": [_fact(f) for f in self.goal_guesses],\n            "object_identities": json.loads(json.dumps(self.object_identities,\n                                                       default=_json_default)),\n        }\n\n    @classmethod\n    def restore(cls, snapshot: dict[str, Any]) -> "GameMemory":\n        """Rebuild from a snapshot. Persistence that cannot be loaded is a log."""\n        return cls(\n            game_id=snapshot["game_id"],\n            level=snapshot["level"],\n            action_semantics={tuple(k): [_unfact(f) for f in v]\n                              for k, v in snapshot["action_semantics"]},\n            mechanics=[_unfact(f) for f in snapshot["mechanics"]],\n            goal_evidence=[_unfact(f) for f in snapshot["goal_evidence"]],\n            counterexamples=[_unfact(f) for f in snapshot["counterexamples"]],\n            winning_paths=[_unpath(p) for p in snapshot["winning_paths"]],\n            current_plan=[tuple(a) for a in snapshot["current_plan"]],\n            level_coordinates=snapshot["level_coordinates"],\n            goal_guesses=[_unfact(f) for f in snapshot["goal_guesses"]],\n            object_identities=snapshot["object_identities"],\n        )\n\n    def apply(self, update: MemoryUpdate,\n              known_transitions: set[int] | None = None) -> MemoryDelta:\n        validate(update, known_transitions)\n        delta = MemoryDelta()\n\n        for name in FACT_LISTS:\n            incoming = getattr(update, name)\n            if not incoming:\n                continue\n            merged, blocked = _merge_facts(getattr(self, name), incoming)\n            delta.blocked.extend(blocked)\n            if merged != getattr(self, name):\n                setattr(self, name, merged)\n                delta.changed.append(name)\n\n        if update.action_semantics:\n            for key, facts in update.action_semantics.items():\n                merged, blocked = _merge_facts(self.action_semantics.get(key, []), facts)\n                delta.blocked.extend(blocked)\n                if merged != self.action_semantics.get(key, []):\n                    self.action_semantics[key] = merged\n                    if "action_semantics" not in delta.changed:\n                        delta.changed.append("action_semantics")\n\n        if update.winning_paths:\n            for path in update.winning_paths:\n                if path not in self.winning_paths:\n                    self.winning_paths.append(path)\n                    if "winning_paths" not in delta.changed:\n                        delta.changed.append("winning_paths")\n\n        for name in ("current_plan", "level_coordinates", "object_identities"):\n            incoming = getattr(update, name)\n            if not incoming:\n                continue\n            setattr(self, name, incoming.copy())\n            delta.changed.append(name)\n\n        delta.contested = sorted(self.contested())\n        if delta.changed:\n            self.unpersisted = True\n        return delta\n\n    def contested(self) -> set[str]:\n        """Statements needing a discriminating probe rather than more assertion."""\n        out = {f.statement for name in FACT_LISTS for f in getattr(self, name)\n               if f.status is Status.CONTESTED}\n        out |= {f.statement for facts in self.action_semantics.values()\n                for f in facts if f.status is Status.CONTESTED}\n        # Two different CONFIRMED claims about one action is also a conflict.\n        for facts in self.action_semantics.values():\n            confirmed = [f.statement for f in facts if f.status is Status.CONFIRMED]\n            if len(confirmed) > 1:\n                out |= set(confirmed)\n        return out\n\n    def replayable(self, level: int, signature: str) -> list[WinningPath]:\n        return [p for p in self.winning_paths if p.usable_from(level, signature)]\n\n    def on_level_transition(self, new_level: int) -> None:\n        """Keep what was learned about the game; drop what was true of the level."""\n        self.level = new_level\n        self.current_plan = []\n        self.level_coordinates = {}\n        self.goal_guesses = []\n        self.object_identities = {}\n        self.unpersisted = True\n\n\ndef _fact(f: Fact) -> dict[str, Any]:\n    return {"statement": f.statement, "status": f.status.value,\n            "evidence": list(f.evidence)}\n\n\ndef _unfact(d: dict[str, Any]) -> Fact:\n    return Fact(d["statement"], Status(d["status"]), tuple(d["evidence"]))\n\n\ndef _path(p: WinningPath) -> dict[str, Any]:\n    return {"level": p.level, "start_signature": p.start_signature,\n            "actions": [list(a) for a in p.actions],\n            "expected_outcomes": list(p.expected_outcomes),\n            "evidence": list(p.evidence)}\n\n\ndef _unpath(d: dict[str, Any]) -> WinningPath:\n    return WinningPath(level=d["level"], start_signature=d["start_signature"],\n                       actions=tuple(tuple(a) for a in d["actions"]),\n                       expected_outcomes=tuple(d["expected_outcomes"]),\n                       evidence=tuple(d["evidence"]))\n\n\nclass MemoryJournal:\n    """Append-only before/after record. Durable per entry, and RESTORABLE.\n\n    An audit log that cannot be loaded is not persistence. `restore` rebuilds the\n    store from the last complete entry, so a run killed mid-game resumes with\n    what it had learned instead of starting over.\n    """\n\n    def __init__(self, path: str | Path) -> None:\n        self.path = Path(path)\n        self.path.parent.mkdir(parents=True, exist_ok=True)\n\n    def record(self, memory: GameMemory, event: str, before: dict[str, Any],\n               delta: MemoryDelta) -> None:\n        entry = {"event": event, "game_id": memory.game_id, "level": memory.level,\n                 "changed": delta.changed, "blocked": delta.blocked,\n                 "contested": delta.contested, "before": before,\n                 "after": memory.snapshot()}\n        with self.path.open("a", encoding="utf-8") as handle:\n            handle.write(json.dumps(entry, separators=(",", ":")) + "\\n")\n            handle.flush()\n            os.fsync(handle.fileno())\n        memory.unpersisted = False\n\n    def entries(self) -> list[dict[str, Any]]:\n        if not self.path.exists():\n            return []\n        out = []\n        for line in self.path.read_text(encoding="utf-8").splitlines():\n            line = line.strip()\n            if not line:\n                continue\n            try:\n                out.append(json.loads(line))\n            except json.JSONDecodeError:\n                continue  # a torn tail costs one entry, not the store\n        return out\n\n    def restore(self) -> GameMemory | None:\n        entries = self.entries()\n        return GameMemory.restore(entries[-1]["after"]) if entries else None\n\n\ndef commit(memory: GameMemory, update: MemoryUpdate, journal: MemoryJournal,\n           known_transitions: set[int] | None = None) -> MemoryDelta:\n    """Apply and persist, in that order, BEFORE the action is taken.\n\n    The action may complete the level, and the transition reset that follows\n    would discard an update that had not yet reached the journal.\n    """\n    before = memory.snapshot()\n    delta = memory.apply(update, known_transitions)\n    journal.record(memory, "update", before, delta)\n    return delta\n\n\ndef transition(memory: GameMemory, new_level: int, journal: MemoryJournal) -> None:\n    """Journal the reset too -- it destroys four fields and must be reconstructible."""\n    before = memory.snapshot()\n    memory.on_level_transition(new_level)\n    journal.record(memory, "level_transition", before,\n                   MemoryDelta(changed=list(VOLATILE)))\n')
(_memory_package / 'duck_memory_adapter.py').write_text('"""Opt-in structured memory for Duck\'s Python-tool agent.\n\nThe memory update is a sibling of ``code`` in the existing Python tool call.\nIt is validated and fsynced by the host *before* the sandbox can execute an\nenvironment action.  Nothing here changes Duck\'s policy when this subclass is\nnot selected.  Winning paths are stored but deliberately never auto-replayed\nin the memory ablation; replay would be a second treatment.\n"""\n\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\nfrom typing import Any\n\nfrom arc3.memory import (\n    Fact,\n    GameMemory,\n    MemoryJournal,\n    MemoryUpdate,\n    Status,\n    WinningPath,\n    commit,\n    transition,\n)\nfrom inference.agent.runtime_state import load_runtime_state\nfrom inference.agent.tool_agent import ToolAgent, _ToolDispatchResult\n\n\nFACT_FIELDS = ("mechanics", "goal_evidence", "counterexamples", "goal_guesses")\nUPDATE_FIELDS = set(FACT_FIELDS) | {\n    "action_semantics", "winning_paths", "current_plan", "level_coordinates",\n    "object_identities",\n}\n\n\ndef _action_key(raw: Any) -> tuple[int, int | None, int | None]:\n    if not isinstance(raw, list) or len(raw) != 3:\n        raise ValueError("an action key must be [id, x, y]")\n    key = tuple(raw)\n    if not isinstance(key[0], int) or isinstance(key[0], bool):\n        raise ValueError("action id must be an integer")\n    if any(value is not None and (not isinstance(value, int) or isinstance(value, bool))\n           for value in key[1:]):\n        raise ValueError("action coordinates must be integers or null")\n    return key\n\n\ndef _fact(raw: Any) -> Fact:\n    if not isinstance(raw, dict) or set(raw) - {"statement", "status", "evidence"}:\n        raise ValueError("a fact must contain only statement, status, and evidence")\n    evidence = raw.get("evidence", [])\n    if not isinstance(evidence, list):\n        raise ValueError("fact evidence must be a list of transition indices")\n    return Fact(\n        statement=raw.get("statement"),\n        status=Status(raw.get("status", "ASSUMED")),\n        evidence=tuple(evidence),\n    )\n\n\ndef parse_memory_update(raw: Any) -> MemoryUpdate:\n    """Convert the JSON tool argument to the tested memory contract."""\n    if not isinstance(raw, dict):\n        raise ValueError("memory_update must be a JSON object")\n    unknown = set(raw) - UPDATE_FIELDS\n    if unknown:\n        raise ValueError(f"unknown memory_update fields: {sorted(unknown)}")\n    values: dict[str, Any] = {}\n    for name in FACT_FIELDS:\n        if name in raw:\n            if not isinstance(raw[name], list):\n                raise ValueError(f"{name} must be a list")\n            values[name] = [_fact(item) for item in raw[name]]\n    if "action_semantics" in raw:\n        if not isinstance(raw["action_semantics"], list):\n            raise ValueError("action_semantics must be a list")\n        semantics: dict[tuple[int, int | None, int | None], list[Fact]] = {}\n        for item in raw["action_semantics"]:\n            if not isinstance(item, dict) or set(item) != {"action", "facts"}:\n                raise ValueError("action_semantics entries need action and facts")\n            key = _action_key(item["action"])\n            if key in semantics or not isinstance(item["facts"], list):\n                raise ValueError("duplicate action or non-list facts")\n            semantics[key] = [_fact(fact) for fact in item["facts"]]\n        values["action_semantics"] = semantics\n    if "winning_paths" in raw:\n        if not isinstance(raw["winning_paths"], list):\n            raise ValueError("winning_paths must be a list")\n        paths = []\n        for item in raw["winning_paths"]:\n            if not isinstance(item, dict) or set(item) != {\n                "level", "start_signature", "actions", "expected_outcomes", "evidence"\n            }:\n                raise ValueError("winning path needs all five fields")\n            if not isinstance(item["actions"], list) or not isinstance(item["expected_outcomes"], list):\n                raise ValueError("winning path actions and outcomes must be lists")\n            if len(item["actions"]) != len(item["expected_outcomes"]):\n                raise ValueError("a winning path needs one expected outcome per action")\n            if not isinstance(item["evidence"], list):\n                raise ValueError("winning path evidence must be a list")\n            paths.append(WinningPath(\n                level=item["level"],\n                start_signature=item["start_signature"],\n                actions=tuple(_action_key(a) for a in item["actions"]),\n                expected_outcomes=tuple(item["expected_outcomes"]),\n                evidence=tuple(item["evidence"]),\n            ))\n        values["winning_paths"] = paths\n    if "current_plan" in raw:\n        if not isinstance(raw["current_plan"], list):\n            raise ValueError("current_plan must be a list")\n        values["current_plan"] = [_action_key(item) for item in raw["current_plan"]]\n    for name in ("level_coordinates", "object_identities"):\n        if name in raw:\n            values[name] = raw[name]\n    return MemoryUpdate(**values)\n\n\ndef known_transition_ids(state_path: Path) -> set[int]:\n    """Match the zero-based ``transitions`` list visible in Duck\'s sandbox."""\n    _, history = load_runtime_state(state_path)\n    return set(range(sum(bool(entry.action.strip()) for entry in history)))\n\n\nclass MemoryToolAgent(ToolAgent):\n    """Duck with explicit, per-game, crash-proof belief revision."""\n\n    def _ensure_session(self, state_path: Path) -> None:\n        super()._ensure_session(state_path)\n        journal_path = state_path.with_suffix(".memory.jsonl")\n        if getattr(self, "_memory_journal_path", None) == journal_path:\n            self._sync_level(state_path)\n            return\n        self._memory_journal_path = journal_path\n        self._memory_journal = MemoryJournal(journal_path)\n        game_id = str(getattr(self, "_gate2_game_id", state_path.stem))\n        restored = self._memory_journal.restore()\n        if restored is not None and restored.game_id != game_id:\n            raise RuntimeError("memory journal belongs to a different game")\n        frame, _ = load_runtime_state(state_path)\n        self._memory = restored or GameMemory(game_id=game_id, level=frame.level if frame else 1)\n        self._memory_commits = 0\n        self._memory_rejections = 0\n        self._memory_transitions = 0\n        self._memory_missing_updates = 0\n        self._memory_empty_updates = 0\n        self._sync_level(state_path)\n\n    def _sync_level(self, state_path: Path) -> None:\n        """Recover a transition even if a prior tool call died after the action."""\n        frame, _ = load_runtime_state(state_path)\n        if frame is not None and frame.level != self._memory.level:\n            transition(self._memory, frame.level, self._memory_journal)\n            self._memory_transitions += 1\n\n    def _tools(self, state_path: Path) -> list[dict[str, Any]]:\n        tools = super()._tools(state_path)\n        tool = tools[0]["function"]\n        tool["description"] += (\n            " Required memory_update is validated and persisted before code runs. "\n            "Use it for concise mechanics and goal evidence, not a turn journal."\n        )\n        tool["parameters"]["properties"]["memory_update"] = {\n            "type": "object",\n            "description": (\n                "Required decision: use {} when nothing new was learned; otherwise "\n                "record at most two concise causal rules or goal hypotheses. "\n                "Fields mechanics, goal_evidence, "\n                "counterexamples, goal_guesses are arrays of facts. A fact is "\n                "{statement, status: ASSUMED|CONFIRMED|REFUTED, evidence: [transition indices]}. "\n                "CONFIRMED and REFUTED require an existing evidence index."\n            ),\n        }\n        tool["parameters"]["required"].append("memory_update")\n        return tools\n\n    def _build_user_prompt(self, action_num: int, **kwargs: Any) -> str:\n        prompt = super()._build_user_prompt(action_num, **kwargs)\n        memory = self._memory\n        facts = []\n        for name in ("mechanics", "goal_evidence", "counterexamples", "goal_guesses"):\n            for fact in getattr(memory, name):\n                facts.append(f"{name}: [{fact.status.value}] {fact.statement}")\n        for action, claims in memory.action_semantics.items():\n            for fact in claims:\n                facts.append(f"action {action}: [{fact.status.value}] {fact.statement}")\n        if memory.current_plan:\n            facts.append(f"current_plan: {memory.current_plan!r}")\n        shown = []\n        shown_chars = 0\n        for fact in facts:\n            if shown_chars + len(fact) + 1 > 3500:\n                break\n            shown.append(fact)\n            shown_chars += len(fact) + 1\n        text = "\\n".join(shown)\n        if len(shown) < len(facts):\n            text += f"\\n[{len(facts) - len(shown)} additional beliefs remain in the per-game journal.]"\n        history = kwargs.get("history_entries") or []\n        latest_id = sum(bool(entry.action.strip()) for entry in history) - 1\n        return prompt + "\\n\\nStructured per-game memory (not a journal of turns):\\n" + (\n            text or "[empty]"\n        ) + (\n            "\\nEvery python tool call must include memory_update as a JSON sibling of code. "\n            "Use {} only when you have learned no new causal rule or goal hypothesis. "\n            "After an action changes the puzzle, consider whether it supports one concise "\n            "mechanic or goal claim; record at most two new claims, not a turn journal. "\n            "For an unverified hypothesis use ASSUMED without evidence, for example "\n            \'{"code":"action([\\\'RIGHT\\\'])",\'\n            \'"memory_update":{"mechanics":[{"statement":"RIGHT moves the frame",\'\n            \'"status":"ASSUMED"}]}}. \'\n            "Use CONFIRMED or REFUTED only with evidence from a transition that already "\n            "exists. Evidence indices are zero-based positions in transitions. "\n            f"The latest available transition index is {latest_id}; never cite a future index. "\n            "To refute one, repeat its exact "\n            "statement as REFUTED with the contradictory transition index. "\n            "At a new level, keep mechanics and verified goal evidence; discard the old plan and coordinates."\n        )\n\n    def _run_python_tool(self, state_path: Path, arguments: dict[str, Any]) -> _ToolDispatchResult:\n        self._ensure_session(state_path)\n        code = str(arguments.get("code", "")).rstrip()\n        if not code:\n            return _ToolDispatchResult(json.dumps({"error": "python requires non-empty code"}))\n        try:\n            compile(code, "<python_tool>", "exec")\n        except SyntaxError as exc:\n            return _ToolDispatchResult(json.dumps({"error": f"Python syntax error: {exc}"}))\n        if "memory_update" not in arguments:\n            self._memory_missing_updates += 1\n        else:\n            try:\n                update = parse_memory_update(arguments["memory_update"])\n                if any(bool(value) for value in vars(update).values()):\n                    commit(\n                        self._memory, update, self._memory_journal,\n                        known_transitions=known_transition_ids(state_path),\n                    )\n                    self._memory_commits += 1\n                else:\n                    self._memory_empty_updates += 1\n            except (TypeError, ValueError) as exc:\n                self._memory_rejections += 1\n                return _ToolDispatchResult(json.dumps({"error": f"memory_update rejected: {exc}"}))\n        result = super()._run_python_tool(state_path, arguments)\n        frame, _ = load_runtime_state(state_path)\n        if frame is not None and frame.level != self._memory.level:\n            transition(self._memory, frame.level, self._memory_journal)\n            self._memory_transitions += 1\n        return result\n')
if str(WORKING_DIR) not in _memory_sys.path:
    _memory_sys.path.insert(0, str(WORKING_DIR))
from arc3.duck_memory_adapter import MemoryToolAgent as _MemoryToolAgent
print('MEMORY_ADAPTER_IMPORTED', flush=True)

(_memory_package / 'scene.py').write_text('"""Lossless, game-agnostic descriptions of ARC colour grids.\n\n``encode_grid`` is intended for storage or programmatic consumption; its small\nrun-length representation remains entirely JSON serializable.  ``render_scene``\nis a compact, human- and LLM-readable view of the same complete board.  Neither\nfunction assigns a meaning to colours or components.\n"""\n\nfrom __future__ import annotations\n\nfrom collections import deque\nfrom numbers import Integral\nfrom typing import Any, Sequence\n\n\nCOLOUR_SYMBOLS = "WwgGcBMP RbSYOrNp".replace(" ", "")\n"""Standard ARC colour symbols, indexed by colour number (0 through 15)."""\n\n\ndef _normalise_grid(grid: Any) -> list[list[int]]:\n    """Validate *grid* and return an independent, plain-Python copy."""\n    if isinstance(grid, (str, bytes)):\n        raise ValueError("grid must be a rectangular sequence of rows")\n    try:\n        rows = list(grid)\n    except TypeError as exc:\n        raise ValueError("grid must be a rectangular sequence of rows") from exc\n    if not 1 <= len(rows) <= 64:\n        raise ValueError("grid height must be between 1 and 64")\n\n    copied: list[list[int]] = []\n    width: int | None = None\n    for row in rows:\n        if isinstance(row, (str, bytes)):\n            raise ValueError("each grid row must be a sequence of colour integers")\n        try:\n            values = list(row)\n        except TypeError as exc:\n            raise ValueError("each grid row must be a sequence of colour integers") from exc\n        if width is None:\n            width = len(values)\n            if not 1 <= width <= 64:\n                raise ValueError("grid width must be between 1 and 64")\n        elif len(values) != width:\n            raise ValueError("grid must be rectangular")\n        clean_row: list[int] = []\n        for value in values:\n            if isinstance(value, bool) or not isinstance(value, Integral):\n                raise ValueError("grid colours must be non-boolean integers from 0 to 15")\n            value = int(value)\n            if not 0 <= value <= 15:\n                raise ValueError("grid colours must be from 0 to 15")\n            clean_row.append(value)\n        copied.append(clean_row)\n    return copied\n\n\ndef _equal_ranges(items: Sequence[Any]) -> list[list[int]]:\n    """Inclusive consecutive ranges of identical items; never merge non-neighbours."""\n    ranges: list[list[int]] = []\n    start = 0\n    for index in range(1, len(items) + 1):\n        if index == len(items) or items[index] != items[start]:\n            ranges.append([start, index - 1])\n            start = index\n    return ranges\n\n\ndef _rle_row(row: Sequence[int]) -> list[list[int]]:\n    runs: list[list[int]] = []\n    start = 0\n    for index in range(1, len(row) + 1):\n        if index == len(row) or row[index] != row[start]:\n            runs.append([row[start], index - start])\n            start = index\n    return runs\n\n\ndef encode_grid(grid: Any) -> dict[str, Any]:\n    """Encode a 1--64 square/rectangular colour grid without losing any cell.\n\n    The payload contains row and column equivalence *runs* for compact rendering,\n    plus RLE for every original row so decoding is independent of those summaries.\n    """\n    cells = _normalise_grid(grid)\n    height, width = len(cells), len(cells[0])\n    columns = [tuple(cells[row][col] for row in range(height)) for col in range(width)]\n    return {\n        "shape": [height, width],\n        "row_ranges": _equal_ranges(cells),\n        "column_ranges": _equal_ranges(columns),\n        "color_grid": [_rle_row(row) for row in cells],\n    }\n\n\ndef decode_grid(payload: Any) -> list[list[int]]:\n    """Decode and validate an ``encode_grid`` payload into a plain nested list."""\n    if not isinstance(payload, dict):\n        raise ValueError("payload must be a mapping")\n    shape = payload.get("shape")\n    encoded_rows = payload.get("color_grid")\n    if (\n        not isinstance(shape, list)\n        or len(shape) != 2\n        or any(isinstance(value, bool) or not isinstance(value, Integral) for value in shape)\n    ):\n        raise ValueError("payload shape must be two integer dimensions")\n    height, width = (int(shape[0]), int(shape[1]))\n    if not 1 <= height <= 64 or not 1 <= width <= 64:\n        raise ValueError("payload shape dimensions must be between 1 and 64")\n    if not isinstance(encoded_rows, list) or len(encoded_rows) != height:\n        raise ValueError("payload color_grid must contain one row per shape row")\n\n    rows: list[list[int]] = []\n    for encoded_row in encoded_rows:\n        if not isinstance(encoded_row, list):\n            raise ValueError("payload row runs must be lists")\n        row: list[int] = []\n        for run in encoded_row:\n            if (\n                not isinstance(run, list)\n                or len(run) != 2\n                or isinstance(run[0], bool)\n                or isinstance(run[1], bool)\n                or not isinstance(run[0], Integral)\n                or not isinstance(run[1], Integral)\n            ):\n                raise ValueError("payload runs must be [colour, positive_count]")\n            colour, count = int(run[0]), int(run[1])\n            if not 0 <= colour <= 15 or count < 1:\n                raise ValueError("payload runs contain an invalid colour or count")\n            row.extend([colour] * count)\n        if len(row) != width:\n            raise ValueError("payload row runs do not match shape width")\n        rows.append(row)\n    return rows\n\n\ndef _components(cells: list[list[int]]) -> list[tuple[int, int, tuple[int, int, int, int]]]:\n    """Return every same-colour 4-connected component as neutral facts."""\n    height, width = len(cells), len(cells[0])\n    seen = [[False] * width for _ in range(height)]\n    found: list[tuple[int, int, tuple[int, int, int, int], int, int]] = []\n    for row in range(height):\n        for col in range(width):\n            if seen[row][col]:\n                continue\n            colour = cells[row][col]\n            seen[row][col] = True\n            queue = deque([(row, col)])\n            area = 0\n            min_row = max_row = row\n            min_col = max_col = col\n            while queue:\n                current_row, current_col = queue.popleft()\n                area += 1\n                min_row, max_row = min(min_row, current_row), max(max_row, current_row)\n                min_col, max_col = min(min_col, current_col), max(max_col, current_col)\n                for next_row, next_col in (\n                    (current_row - 1, current_col), (current_row + 1, current_col),\n                    (current_row, current_col - 1), (current_row, current_col + 1),\n                ):\n                    if (\n                        0 <= next_row < height and 0 <= next_col < width\n                        and not seen[next_row][next_col]\n                        and cells[next_row][next_col] == colour\n                    ):\n                        seen[next_row][next_col] = True\n                        queue.append((next_row, next_col))\n            found.append((colour, area, (min_row, min_col, max_row, max_col), row, col))\n    found.sort(key=lambda item: (-item[1], item[3], item[4], item[0]))\n    return [(colour, area, bbox) for colour, area, bbox, _, _ in found]\n\n\ndef _range_text(ranges: Sequence[Sequence[int]]) -> str:\n    return ", ".join(str(start) if start == end else f"{start}-{end}" for start, end in ranges)\n\n\ndef render_scene(grid: Any, previous: Any | None = None) -> str:\n    """Render a complete, compact SCENE_V1 snapshot and neutral change facts."""\n    cells = _normalise_grid(grid)\n    height, width = len(cells), len(cells[0])\n    encoded = encode_grid(cells)\n    row_ranges = encoded["row_ranges"]\n    column_ranges = encoded["column_ranges"]\n    reduced_rows = [\n        "".join(COLOUR_SYMBOLS[cells[row_range[0]][column_range[0]]] for column_range in column_ranges)\n        for row_range in row_ranges\n    ]\n\n    lines = [\n        "SCENE_V1",\n        f"shape: {height}x{width}",\n        "legend: 0=W 1=w 2=g 3=G 4=c 5=B 6=M 7=P 8=R 9=b 10=S 11=Y 12=O 13=r 14=N 15=p",\n        f"rows (inclusive original ranges): {_range_text(row_ranges)}",\n        f"columns (inclusive original ranges): {_range_text(column_ranges)}",\n        "reduced letter grid (row-ranges x column-ranges):",\n        *reduced_rows,\n    ]\n\n    if previous is None:\n        lines.append("changes: full snapshot (no previous grid)")\n    else:\n        before = _normalise_grid(previous)\n        if (len(before), len(before[0])) != (height, width):\n            lines.append(\n                f"changes: shape-change {len(before)}x{len(before[0])} -> {height}x{width}; full current snapshot above"\n            )\n        else:\n            changed = [\n                (row, col, cells[row][col])\n                for row in range(height)\n                for col in range(width)\n                if before[row][col] != cells[row][col]\n            ]\n            if len(changed) <= 128:\n                facts = ", ".join(f"({row},{col})={COLOUR_SYMBOLS[colour]}" for row, col, colour in changed)\n                lines.append(f"changes: {len(changed)} exact cells" + (f": {facts}" if facts else ""))\n            else:\n                min_row = min(row for row, _, _ in changed)\n                max_row = max(row for row, _, _ in changed)\n                min_col = min(col for _, col, _ in changed)\n                max_col = max(col for _, col, _ in changed)\n                lines.append(\n                    f"changes: {len(changed)} cells; bbox inclusive ({min_row},{min_col})-({max_row},{max_col}); "\n                    "exact change list omitted; full current snapshot above"\n                )\n\n    components = _components(cells)\n    lines.append(f"components (4-connected, all colours): {len(components)} total")\n    for colour, area, (min_row, min_col, max_row, max_col) in components[:64]:\n        lines.append(\n            f"- color {colour}={COLOUR_SYMBOLS[colour]} area={area} bbox=({min_row},{min_col})-({max_row},{max_col})"\n        )\n    if len(components) > 64:\n        lines.append("component summary truncated after 64; lossless grid remains complete above")\n    return "\\n".join(lines)\n')

(_memory_package / 'duck_scene_adapter.py').write_text('"""Supply a complete deterministic observation before Duck requests a tool."""\n\nfrom arc3.scene import render_scene\n\n\nSCENE_GUIDANCE = """\n\nComplete scene observations:\n- Every user turn includes SCENE_V1, a complete lossless encoding of the current\n  visible board. Read it directly before deciding whether more inspection is needed.\n- You may read the supplied scene in full. Restrictions on printing full boards\n  concern redundant Python tool output, not reading this supplied observation.\n- Repeated adjacent rows and columns are compressed only when their entire\n  contents are identical. The row/column ranges preserve original coordinates;\n  they are not inferred game tiles. Convert any target back to original row/col.\n- Components describe observed geometry only. Background, HUD, targets, controls,\n  and rules are hypotheses for you to test; the renderer assigns none of them.\n- Changes are relative to the previous model-turn observation, which may span\n  several real actions. They are not evidence for one action when a batch ran.\n- The complete current snapshot remains authoritative. Python is available for\n  calculations, small crops, hypotheses, search and actions. You do not need to\n  spend an inspection call rebuilding the scene already provided.\n"""\n\n\nclass SceneObservationMixin:\n    """Mixin preceding ToolAgent; no modification of control instances."""\n\n    def __init__(self, *args, **kwargs):\n        super().__init__(*args, **kwargs)\n        self._scene_previous_grid = None\n        self._scene_previous_level = None\n        self._scene_observations = 0\n        self._scene_chars = 0\n        self._system_prompt += SCENE_GUIDANCE\n\n    def _build_user_prompt(self, action_num, *, current_frame=None, **kwargs):\n        prompt = super()._build_user_prompt(\n            action_num, current_frame=current_frame, **kwargs\n        )\n        prompt = prompt.replace(\n            "Only letter-coded board views and lightweight metadata are exposed; raw numeric color IDs are not available.",\n            "The supplied scene includes letter-coded cells, color IDs and exact original coordinates; Python retains its usual board views.",\n        )\n        if current_frame is None:\n            raise ValueError("Scene agent requires an observable current frame")\n        grid = tuple(tuple(row) for row in current_frame.grid)\n        previous = self._scene_previous_grid\n        level_note = ""\n        if self._scene_previous_level != current_frame.level:\n            previous = None\n            level_note = "New level observation; infer or recheck the goal and rules.\\n"\n        scene = render_scene(grid, previous=previous)\n        self._scene_previous_grid = grid\n        self._scene_previous_level = current_frame.level\n        self._scene_observations += 1\n        self._scene_chars += len(scene)\n        return prompt + "\\n\\n" + level_note + scene\n\n\ndef scene_delivery_problems(rows):\n    """Validate actual user-message evidence, not an intended environment flag."""\n    problems = []\n    for row in rows:\n        arm = row["trial_id"]\n        label = f"{arm}/{row[\'game_id\']}"\n        scene = arm in ("C", "T")\n        image = arm != "T"\n        for field in ("scene_observations", "scene_chars", "scene_delivered_messages"):\n            if (int(row.get(field, 0)) > 0) != scene:\n                problems.append(f"{label}: unexpected {field}={row.get(field)}")\n        images = int(row.get("scene_image_messages", 0))\n        texts = int(row.get("scene_text_only_messages", 0))\n        if scene:\n            observations = int(row.get("scene_observations", 0))\n            delivered = int(row.get("scene_delivered_messages", 0))\n            expected_modality = images if image else texts\n            if not (observations == delivered == expected_modality):\n                problems.append(\n                    f"{label}: incomplete per-turn delivery "\n                    f"observations={observations}, delivered={delivered}, "\n                    f"expected_modality={expected_modality}"\n                )\n        if image and not (images > 0 and texts == 0):\n            problems.append(f"{label}: expected image-only turns, saw {images}/{texts}")\n        if not image and not (images == 0 and texts > 0):\n            problems.append(f"{label}: expected text-only turns, saw {images}/{texts}")\n    return problems\n')
from arc3.duck_scene_adapter import SceneObservationMixin, scene_delivery_problems

(_memory_package / 'duck_gptoss_adapter.py').write_text('"""Text-only, native-function transport for a Duck scene agent using GPT-OSS.\n\nThis mixin is deliberately opt-in.  Put it before ``SceneObservationMixin``\nand Duck\'s ``ToolAgent`` in the MRO.  It does not alter the action loop,\ntool limits, or the scene renderer; it only adapts the messages and request\npayload crossing the model boundary.\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport os\nimport time\nfrom pathlib import Path\nfrom typing import Any\n\nimport requests\n\nfrom inference.agent import tool_agent as _duck_tool_agent\nfrom inference.agent.prompts import MULTIMODAL_CONTEXT_ADDENDUM, TOOL_CALL_FORMAT_GUIDANCE\n\n\nNATIVE_FUNCTION_CALL_GUIDANCE = (\n    "Call `python` through the native function-calling interface when you need "\n    "to use it. Do not write XML, tags, or a textual tool-call wrapper in "\n    "assistant content or reasoning."\n)\n\n_HTTP_ERROR_BODY_LIMIT = 4096\n\n\ndef _native_prompt(prompt: str) -> str:\n    """Remove Duck\'s model-specific call-format wording without touching gameplay rules."""\n    return (\n        prompt.replace(MULTIMODAL_CONTEXT_ADDENDUM, "")\n        .replace(TOOL_CALL_FORMAT_GUIDANCE, NATIVE_FUNCTION_CALL_GUIDANCE)\n    )\n\n\ndef _require_text_only_messages(messages: list[dict[str, Any]]) -> None:\n    """Fail closed if a caller would send a vision/content-part payload to GPT-OSS."""\n    for index, message in enumerate(messages):\n        content = message.get("content")\n        if content is None or isinstance(content, str):\n            continue\n        raise ValueError(\n            "GPT-OSS native transport requires text-only messages; "\n            f"message {index} has non-text content."\n        )\n\n\ndef _native_messages(messages: list[dict[str, Any]]) -> list[dict[str, Any]]:\n    """Copy message envelopes while replacing only inherited format guidance."""\n    normalized: list[dict[str, Any]] = []\n    for message in messages:\n        copied = dict(message)\n        if isinstance(copied.get("content"), str):\n            copied["content"] = _native_prompt(copied["content"])\n        normalized.append(copied)\n    return normalized\n\n\ndef _contains_xml_tool_markup(message: dict[str, Any]) -> bool:\n    for field in ("content", "reasoning", "reasoning_content"):\n        value = message.get(field)\n        if isinstance(value, str) and ("<tool_call" in value.lower() or "<function=" in value.lower()):\n            return True\n    return False\n\n\nclass GPTOSSNativeFunctionMixin:\n    """Adapt an existing Duck + scene agent to GPT-OSS function transport.\n\n    Assumes Duck\'s current ``ToolAgent.analyze`` continues to execute native\n    ``tool_calls`` and append OpenAI ``role=\'tool\'`` result messages with the\n    corresponding ``tool_call_id``.  The mixin intentionally leaves that\n    action/time-limit logic untouched.\n    """\n\n    def __init__(self, *args: Any, **kwargs: Any) -> None:\n        super().__init__(*args, **kwargs)\n        self._system_prompt = _native_prompt(self._system_prompt)\n\n    def _build_user_prompt(self, action_num: int, **kwargs: Any) -> str:\n        return _native_prompt(super()._build_user_prompt(action_num, **kwargs))\n\n    def _build_user_message(self, user_prompt: str, current_frame: Any) -> dict[str, Any]:\n        """Send the complete SceneObservationMixin prompt as text, never process an image.\n\n        The concrete GPT-OSS class must instrument this method directly: the\n        base implementation may construct an image attachment before returning,\n        which would make a base-only modality counter describe the wrong\n        payload boundary.\n        """\n        del current_frame\n        return {"role": "user", "content": _native_prompt(user_prompt)}\n\n    def _chat_completion(\n        self,\n        messages: list[dict[str, Any]],\n        *,\n        tools: list[dict[str, Any]] | None,\n        request_timeout_seconds: float | None = None,\n    ) -> Any:\n        """Use standard OpenAI tools/tool_calls without Qwen vLLM template options."""\n        messages = _native_messages(messages)\n        _require_text_only_messages(messages)\n        payload: dict[str, Any] = {\n            "model": self._model.model_id,\n            "messages": messages,\n            "stream": False,\n            "temperature": _duck_tool_agent._LOCAL_ANALYZER_TEMPERATURE,\n            "top_p": _duck_tool_agent._LOCAL_ANALYZER_TOP_P,\n            "reasoning_effort": "high",\n            "ignore_eos": False,\n        }\n        if self._max_output_tokens is not None:\n            payload["max_tokens"] = self._max_output_tokens\n        if _duck_tool_agent._LOCAL_ANALYZER_SEED >= 0:\n            payload["seed"] = _duck_tool_agent._LOCAL_ANALYZER_SEED\n        if tools:\n            payload["tools"] = tools\n            # GPT-OSS\'s supported native function mode is auto.  Do not inherit\n            # Duck\'s model-specific required/named-tool selection here.\n            payload["tool_choice"] = "auto"\n\n        sequence = int(getattr(self, "_gptoss_http_sequence", 0)) + 1\n        self._gptoss_http_sequence = sequence\n        self._gptoss_audit("request", sequence, payload)\n\n        response = requests.post(\n            f"{self._model.base_url.rstrip(\'/\')}/chat/completions",\n            headers=self._headers(),\n            json=payload,\n            timeout=(\n                request_timeout_seconds\n                if request_timeout_seconds is not None\n                else self._timeout\n            ),\n        )\n        try:\n            response.raise_for_status()\n        except requests.HTTPError as exc:\n            detail = str(response.text or "")[:_HTTP_ERROR_BODY_LIMIT].strip()\n            message = f"{exc}"\n            if detail:\n                message += f" | response: {detail}"\n            raise requests.RequestException(message) from exc\n        if getattr(response, "status_code", 200) >= 400:\n            detail = str(response.text or "")[:_HTTP_ERROR_BODY_LIMIT].strip()\n            message = f"{response.status_code} Error"\n            if detail:\n                message += f" | response: {detail}"\n            raise requests.RequestException(message)\n        response_payload = response.json()\n        self._gptoss_audit("response", sequence, response_payload)\n        choices = response_payload.get("choices", [])\n        if not choices:\n            raise requests.RequestException("server returned no choices")\n        choice = choices[0]\n        message = choice.get("message", {})\n        if not message.get("tool_calls") and _contains_xml_tool_markup(message):\n            # Duck\'s base agent has a legacy XML recovery fallback.  Refuse XML\n            # here so GPT-OSS cannot enter that parser path; native tool_calls\n            # are the only accepted transport for this adapter.\n            raise requests.RequestException(\n                "GPT-OSS response used unsupported XML tool markup; expected native tool_calls."\n            )\n        return _duck_tool_agent._ChatCompletionResult(\n            message=message,\n            finish_reason=str(choice.get("finish_reason", "") or ""),\n            usage=response_payload.get("usage"),\n        )\n\n    def _gptoss_audit(self, event: str, sequence: int, payload: dict[str, Any]) -> None:\n        """Persist the actual normalized HTTP body, never authorization headers.\n\n        Opt-in for bounded development runs; the factory supplies a per-game\n        path. Write failure stops the request rather than claiming evidence\n        that was never preserved.\n        """\n        audit_path = getattr(self, "_gptoss_audit_path", None)\n        if audit_path is None:\n            return\n        path = Path(audit_path)\n        path.parent.mkdir(parents=True, exist_ok=True)\n        record = {"event": event, "sequence": sequence, "epoch": time.time(), "payload": payload}\n        with path.open("a", encoding="utf-8") as handle:\n            handle.write(json.dumps(record, sort_keys=True) + "\\n")\n            handle.flush()\n            os.fsync(handle.fileno())\n')
from arc3.duck_gptoss_adapter import GPTOSSNativeFunctionMixin


In [ ]:
# GPT-OSS lifecycle smoke; all previous Qwen smoke approvals are invalid for this model.
GATE2_COMPARISON_SMOKE = True
GATE2_SMOKE_GAME_SECONDS = 360.0  # T1: need wall > first gameplay completion
GATE2_TRIAL_SECONDS = 600.0
GATE2_SMOKE_STARTUP_LIMIT_SECONDS = 900.0
GATE2_SAFETY_MARGIN_SECONDS = 4860.0
GATE2_SOFT_STOP_GRACE_SECONDS = 120.0
GATE2_CONTROL_CAP = 0
GATE2_CANDIDATE_CAP = 0
GATE2_CONCURRENCY = 1 if GATE2_COMPARISON_SMOKE else 3
GATE2_ACTION_CAP = 400
GATE2_SEEDS = (1214842320, 656940509)

if TRUE_SUBMISSION:
    raise RuntimeError("Gate 2 comparison is local-only and must never be submitted.")
if float(getattr(target, "max_runtime_s", 0.0) or 0.0) != 32400.0:
    raise RuntimeError(
        f"Expected the 32400-second notebook budget, got {target.max_runtime_s!r}."
    )
if GATE2_SAFETY_MARGIN_SECONDS < 0.15 * float(target.max_runtime_s):
    raise RuntimeError("Gate 2 safety margin is below 15% of the notebook budget.")
if GATE1_SERVER_READY_EPOCH <= NOTEBOOK_START_EPOCH:
    raise RuntimeError("Server-ready timestamp must follow notebook start.")
if GATE1_SERVER_STARTUP_SECONDS > GATE2_SMOKE_STARTUP_LIMIT_SECONDS:
    raise TimeoutError(
        f"vLLM startup exceeded {GATE2_SMOKE_STARTUP_LIMIT_SECONDS}s: "
        f"{GATE1_SERVER_STARTUP_SECONDS}s."
    )

import threading as _gate2_threading
import inference.agent.tool_agent as _gate2_tool_agent_module
from inference.agent.tool_agent import ToolAgent as _Gate2ToolAgent

class _GptOssToolAgent(GPTOSSNativeFunctionMixin, SceneObservationMixin, _Gate2ToolAgent):
    pass

from inference.framework.solver import _HarnessGameSession as _Gate2Session

_GATE2_METRICS_LOCK = _gate2_threading.Lock()
_GATE2_METRICS = {
    "instrumentation_epoch": time.time(),
    "first_request_started_epoch": None,
    "first_response_epoch": None,
    "llm_calls_total": 0,
    "finish_reason_length_total": 0,
}
_GATE2_SESSION_METRICS = {}

if not getattr(_GptOssToolAgent, "_gate2_call_counter_installed", False):
    _GATE2_ORIGINAL_CHAT_COMPLETION = _GptOssToolAgent._chat_completion

    def _gate2_counted_chat_completion(self, *args, **kwargs):
        is_first = False
        started_epoch = time.time()
        with _GATE2_METRICS_LOCK:
            self._gate2_llm_calls = int(getattr(self, "_gate2_llm_calls", 0)) + 1
            _GATE2_METRICS["llm_calls_total"] += 1
            if _GATE2_METRICS["first_request_started_epoch"] is None:
                _GATE2_METRICS["first_request_started_epoch"] = started_epoch
                is_first = True
        try:
            result = _GATE2_ORIGINAL_CHAT_COMPLETION(self, *args, **kwargs)
            finish_reason = str(getattr(result, "finish_reason", "") or "").lower()
            if finish_reason == "length":
                with _GATE2_METRICS_LOCK:
                    self._gate2_finish_reason_length = int(
                        getattr(self, "_gate2_finish_reason_length", 0)
                    ) + 1
                    _GATE2_METRICS["finish_reason_length_total"] += 1
            return result
        finally:
            if is_first:
                with _GATE2_METRICS_LOCK:
                    _GATE2_METRICS["first_response_epoch"] = time.time()

    _GptOssToolAgent._chat_completion = _gate2_counted_chat_completion
    _GptOssToolAgent._gate2_call_counter_installed = True

if not getattr(_Gate2Session, "_gate2_action_counter_installed", False):
    _GATE2_ORIGINAL_EXECUTE_ACTION = _Gate2Session._execute_action

    def _gate2_counted_execute_action(self, *args, **kwargs):
        payload = _GATE2_ORIGINAL_EXECUTE_ACTION(self, *args, **kwargs)
        self._gate2_executed_actions = int(getattr(self, "_gate2_executed_actions", 0)) + 1
        if not bool(payload.get("board_changed")):
            self._gate2_noop_actions = int(getattr(self, "_gate2_noop_actions", 0)) + 1
        return payload

    _Gate2Session._execute_action = _gate2_counted_execute_action
    _Gate2Session._gate2_action_counter_installed = True

if not getattr(_Gate2Session, "_gate2_session_timer_installed", False):
    _GATE2_ORIGINAL_SESSION_PLAY = _Gate2Session.play

    def _gate2_traced_session_play(self):
        active_started = time.monotonic()
        try:
            return _GATE2_ORIGINAL_SESSION_PLAY(self)
        finally:
            run = getattr(self.game, "game_run", None)
            game_id = getattr(run, "game_id", str(self.game_index))
            trial_id = str(getattr(self.analyzer, "_gate2_trial_id", "unknown"))
            row = {
                "session_started": True,
                "active_wall_seconds": time.monotonic() - active_started,
                "llm_calls": int(getattr(self.analyzer, "_gate2_llm_calls", 0)),
                "finish_reason_length": int(
                    getattr(self.analyzer, "_gate2_finish_reason_length", 0)
                ),
                "no_op_actions": int(getattr(self, "_gate2_noop_actions", 0)),
                "instrumented_actions": int(getattr(self, "_gate2_executed_actions", 0)),
                "scene_observations": int(getattr(self.analyzer, "_scene_observations", 0)),
                "scene_chars": int(getattr(self.analyzer, "_scene_chars", 0)),
                "scene_delivered_messages": int(getattr(self.analyzer, "_scene_delivered_messages", 0)),
                "scene_image_messages": int(getattr(self.analyzer, "_scene_image_messages", 0)),
                "scene_text_only_messages": int(getattr(self.analyzer, "_scene_text_only_messages", 0)),
                "memory_commits": int(getattr(self.analyzer, "_memory_commits", 0)),
                "memory_rejections": int(getattr(self.analyzer, "_memory_rejections", 0)),
                "memory_missing_updates": int(getattr(self.analyzer, "_memory_missing_updates", 0)),
                "memory_empty_updates": int(getattr(self.analyzer, "_memory_empty_updates", 0)),
                "memory_transitions": int(getattr(self.analyzer, "_memory_transitions", 0)),
            }
            with _GATE2_METRICS_LOCK:
                _GATE2_SESSION_METRICS[(trial_id, game_id)] = row

    _Gate2Session.play = _gate2_traced_session_play
    _Gate2Session._gate2_session_timer_installed = True

print(
    "GATE2_SETTINGS "
    + json.dumps(
        {
            "smoke": GATE2_COMPARISON_SMOKE,
            "trial_seconds": (
                GATE2_SMOKE_GAME_SECONDS if GATE2_COMPARISON_SMOKE else GATE2_TRIAL_SECONDS
            ),
            "concurrency": GATE2_CONCURRENCY,
            "action_cap": GATE2_ACTION_CAP,
            "control_cap": GATE2_CONTROL_CAP,
            "candidate_cap": GATE2_CANDIDATE_CAP,
            "seeds": list(GATE2_SEEDS),
            "safety_margin_seconds": GATE2_SAFETY_MARGIN_SECONDS,
        },
        sort_keys=True,
    ),
    flush=True,
)

# Observe actual message shape at the provider boundary for every arm.
_scene_original_build_message = _GptOssToolAgent._build_user_message
def _scene_counted_build_message(self, user_prompt, current_frame):
    message = _scene_original_build_message(self, user_prompt, current_frame)
    content = message.get("content", "")
    parts = content if isinstance(content, list) else [{"type": "text", "text": content}]
    has_image = any(p.get("type") == "image_url" for p in parts)
    has_scene = any(p.get("type") == "text" and "SCENE_V1" in str(p.get("text", "")) for p in parts)
    counter = "_scene_image_messages" if has_image else "_scene_text_only_messages"
    setattr(self, counter, int(getattr(self, counter, 0)) + 1)
    if has_scene:
        self._scene_delivered_messages = int(getattr(self, "_scene_delivered_messages", 0)) + 1
    return message
_GptOssToolAgent._build_user_message = _scene_counted_build_message


In [ ]:
# Gate 2 crash-proof paired comparison. Every arm shares this one vLLM session.
import asyncio as _gate2_asyncio
import copy as _gate2_copy
import os as _gate2_os
import signal as _gate2_signal


def _offline_games(env_dir: str):
    import arc_agi
    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.OFFLINE,
        environments_dir=env_dir,
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.OFFLINE,
        environments_dir=env_dir,
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError(f"No offline environments found under {env_dir}.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


PUBLIC_GAME_IDS = (
    "tn36-ef4dde99", "lf52-271a04aa", "cn04-2fe56bfb", "bp35-0a0ad940",
    "wa30-ee6fef47", "lp85-305b61c3", "r11l-495a7899", "tu93-0768757b",
    "sp80-589a99af", "m0r0-492f87ba", "vc33-5430563c", "ar25-0c556536",
    "ka59-38d34dbb", "sc25-635fd71a", "sk48-d8078629", "dc22-fdcac232",
    "cd82-fb555c5d", "ft09-0d8bbf25", "g50t-5849a774", "ls20-9607627b",
    "re86-8af5384d", "s5i5-18d95033", "sb26-7fbdac44", "su15-1944f8ab",
    "tr87-cd924810",
)
GATE2_GAME_IDS = ("r11l-495a7899",)

competition_env_files = str(
    Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels").parent
    / "environment_files"
)
offline_games = _offline_games(competition_env_files)
offline_by_id = {game.env_name: game for game in offline_games}
if len(offline_by_id) != len(offline_games):
    raise RuntimeError("The offline public game list contains duplicate IDs.")
missing_public = sorted(set(PUBLIC_GAME_IDS) - set(offline_by_id))
extra_public = sorted(set(offline_by_id) - set(PUBLIC_GAME_IDS))
if missing_public or extra_public:
    raise RuntimeError(
        f"Offline public game set changed; missing={missing_public}, extra={extra_public}."
    )

_gate2_expected_ids = list(GATE2_GAME_IDS[:1] if GATE2_COMPARISON_SMOKE else GATE2_GAME_IDS)
_gate2_trial_seconds = (
    GATE2_SMOKE_GAME_SECONDS if GATE2_COMPARISON_SMOKE else GATE2_TRIAL_SECONDS
)
_gate2_trial_specs = [
    {"trial_id": "G", "replicate": 0, "arm": "G", "cap": 0, "seed": GATE2_SEEDS[0]},
]

print(
    "GATE2_PROTOCOL "
    + json.dumps(
        {
            "games": _gate2_expected_ids,
            "trial_seconds": _gate2_trial_seconds,
            "concurrency": GATE2_CONCURRENCY,
            "trials": _gate2_trial_specs,
            "note": "Budgets are matched; realized token counts are measured, not forced equal.",
        },
        sort_keys=True,
    ),
    flush=True,
)

_GATE2_ROOT = WORKING_DIR / "gate2-comparison"
_GATE2_ROOT.mkdir(parents=True, exist_ok=True)
_GATE2_PROGRESS_START = time.monotonic()
_gate2_hard_guard_triggered = False
_gate2_benchmark_error = None
_gate2_benchmark_ok = False
_gate2_teardown_error = None
_gate2_teardown_ok = False
_gate2_teardown_attempts = []
_gate2_teardown_result = {}
_gate2_post_gpu_rows = []
_gate2_completed_trials = []
_gate2_trial_benchmarks = {}


def _gate2_atomic_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(value, indent=2, sort_keys=True) + "\n")
    _gate2_os.replace(tmp, path)


def _gate2_ratio(numerator, denominator):
    return float(numerator) / float(denominator) if denominator else None


def _gate2_run_snapshot(trial_id, run, trial_started):
    session = _GATE2_SESSION_METRICS.get((trial_id, run.game_id)) or {}
    generated_tokens = sum(int(record.generated_tokens) for record in run.history)
    generated_tokens += int(getattr(run, "final_generated_tokens", 0) or 0)
    base_raw = run.base_actions_per_level
    return {
        "trial_id": trial_id,
        "game_id": run.game_id,
        "state": str(run.state),
        "final_score": float(run.final_score) if run.final_score is not None else None,
        "actions_taken": len(run.history),
        "actions_per_level": [int(value) for value in run.actions_per_level],
        "base_actions_per_level": (
            None if base_raw is None else [int(value) for value in base_raw]
        ),
        "levels_completed": int(run.levels_completed),
        "number_of_levels": int(run.number_of_levels),
        "generated_tokens": generated_tokens,
        "llm_calls": int(session.get("llm_calls", 0)),
        "finish_reason_length": int(session.get("finish_reason_length", 0)),
        "no_op_actions": int(session.get("no_op_actions", 0)),
        "instrumented_actions": int(session.get("instrumented_actions", 0)),
        "scene_observations": int(session.get("scene_observations", 0)),
        "scene_chars": int(session.get("scene_chars", 0)),
        "scene_delivered_messages": int(session.get("scene_delivered_messages", 0)),
        "scene_image_messages": int(session.get("scene_image_messages", 0)),
        "scene_text_only_messages": int(session.get("scene_text_only_messages", 0)),
        "memory_commits": int(session.get("memory_commits", 0)),
        "memory_rejections": int(session.get("memory_rejections", 0)),
        "memory_missing_updates": int(session.get("memory_missing_updates", 0)),
        "memory_empty_updates": int(session.get("memory_empty_updates", 0)),
        "memory_transitions": int(session.get("memory_transitions", 0)),
        "active_wall_seconds": session.get("active_wall_seconds"),
        "trial_elapsed_seconds": time.monotonic() - trial_started,
        "notebook_elapsed_seconds": time.monotonic() - _GATE2_PROGRESS_START,
    }


def _gate2_persist_run(trial_id, run, trial_started, terminal):
    row = _gate2_run_snapshot(trial_id, run, trial_started)
    row["terminal"] = bool(terminal)
    _gate2_atomic_json(_GATE2_ROOT / trial_id / "games" / f"{run.game_id}.json", row)
    return row


def _gate2_trial_rows(trial_id):
    rows = []
    for path in sorted((_GATE2_ROOT / trial_id / "games").glob("*.json")):
        try:
            rows.append(json.loads(path.read_text()))
        except Exception as exc:
            print(f"GATE2_PROGRESS_READ_ERROR path={path} error={exc!r}", flush=True)
    return rows


def _gate2_write_progress_index():
    rows = []
    for spec in _gate2_trial_specs:
        rows.extend(_gate2_trial_rows(spec["trial_id"]))
    _gate2_atomic_json(_GATE2_ROOT / "progress.json", rows)
    jsonl_tmp = _GATE2_ROOT / "progress.jsonl.tmp"
    jsonl_tmp.write_text("".join(json.dumps(row, sort_keys=True) + "\n" for row in rows))
    _gate2_os.replace(jsonl_tmp, _GATE2_ROOT / "progress.jsonl")
    return rows


def _gate2_flush_trial(trial_id, trial_bm, trial_started):
    for run in list(trial_bm.game_runs):
        terminal = str(run.state) != "playing"
        _gate2_persist_run(trial_id, run, trial_started, terminal=terminal)
    rows = _gate2_write_progress_index()
    try:
        trial_bm._save_json()
    except Exception as exc:
        print(
            f"GATE2_BENCHMARK_SAVE_ERROR trial={trial_id} error={exc!r}",
            flush=True,
        )
    return rows


async def _gate2_progress_loop(trial_id, trial_bm, trial_started, stop_event):
    emitted = set()
    try:
        while not stop_event.is_set():
            for run in list(trial_bm.game_runs):
                if run.game_id in emitted or str(run.state) == "playing":
                    continue
                row = _gate2_persist_run(trial_id, run, trial_started, terminal=True)
                _gate2_write_progress_index()
                try:
                    trial_bm._save_json()
                except Exception as exc:
                    print(
                        f"GATE2_INCREMENTAL_SAVE_ERROR trial={trial_id} error={exc!r}",
                        flush=True,
                    )
                emitted.add(run.game_id)
                print(
                    "GATE2_GAME_COMPLETE "
                    + json.dumps(
                        {
                            "trial_id": trial_id,
                            "game_id": row["game_id"],
                            "state": row["state"],
                            "levels_completed": row["levels_completed"],
                            "actions_taken": row["actions_taken"],
                            "generated_tokens": row["generated_tokens"],
                            "trial_elapsed_seconds": row["trial_elapsed_seconds"],
                        },
                        sort_keys=True,
                    ),
                    flush=True,
                )
            await _gate2_asyncio.sleep(0.25)
    finally:
        for run in list(trial_bm.game_runs):
            if run.game_id not in emitted and str(run.state) != "playing":
                row = _gate2_persist_run(trial_id, run, trial_started, terminal=True)
                print(
                    "GATE2_GAME_COMPLETE "
                    + json.dumps(
                        {
                            "trial_id": trial_id,
                            "game_id": row["game_id"],
                            "state": row["state"],
                            "levels_completed": row["levels_completed"],
                            "actions_taken": row["actions_taken"],
                            "generated_tokens": row["generated_tokens"],
                            "trial_elapsed_seconds": row["trial_elapsed_seconds"],
                        },
                        sort_keys=True,
                    ),
                    flush=True,
                )
        _gate2_write_progress_index()


class _SceneToolAgent(SceneObservationMixin, _Gate2ToolAgent):
    pass


def _gate2_make_analyzer_factory(spec, solver):
    trial_id = spec["trial_id"]

    def factory(game, index):
        analyzer_cls = _GptOssToolAgent
        analyzer = analyzer_cls(
            model=os.environ["LOCAL_ANALYZER_MODEL_ID"],
            timeout=solver.analyzer_timeout,
            save_request_logs=solver.save_request_logs,
        )
        analyzer._gate2_trial_id = trial_id
        analyzer._gate2_game_id = getattr(game, "env_name", str(index))
        analyzer._gptoss_audit_path = (_GATE2_ROOT / trial_id / "http" / f"{analyzer._gate2_game_id}.jsonl")
        return analyzer

    return factory


def _gate2_summarize_trial(spec, rows, gpu_seconds):
    actions = sum(int(row["actions_taken"]) for row in rows)
    calls = sum(int(row["llm_calls"]) for row in rows)
    tokens = sum(int(row["generated_tokens"]) for row in rows)
    levels = sum(int(row["levels_completed"]) for row in rows)
    no_ops = sum(int(row["no_op_actions"]) for row in rows)
    length_count = sum(int(row["finish_reason_length"]) for row in rows)
    score = sum(float(row["final_score"] or 0.0) for row in rows) / len(_gate2_expected_ids)
    return {
        **spec,
        "game_count": len(rows),
        "gpu_seconds": gpu_seconds,
        "weighted_rhae": score,
        "weighted_rhae_per_gpu_second": _gate2_ratio(score, gpu_seconds),
        "generated_tokens": tokens,
        "llm_calls": calls,
        "actions": actions,
        "completed_levels": levels,
        "games_with_progress": sum(int(row["levels_completed"] > 0) for row in rows),
        "games_won": sum(int(row["state"] == "won") for row in rows),
        "tokens_per_call": _gate2_ratio(tokens, calls),
        "actions_per_call": _gate2_ratio(actions, calls),
        "tokens_per_action": _gate2_ratio(tokens, actions),
        "tokens_per_completed_level": _gate2_ratio(tokens, levels),
        "no_op_actions": no_ops,
        "no_op_rate": _gate2_ratio(no_ops, actions),
        "finish_reason_length_count": length_count,
        "memory_commits": sum(int(row["memory_commits"]) for row in rows),
        "memory_rejections": sum(int(row["memory_rejections"]) for row in rows),
        "memory_missing_updates": sum(int(row["memory_missing_updates"]) for row in rows),
        "memory_empty_updates": sum(int(row["memory_empty_updates"]) for row in rows),
        "memory_transitions": sum(int(row["memory_transitions"]) for row in rows),
        "finish_reason_length_rate": _gate2_ratio(length_count, calls),
    }


def _gate2_teardown_with_recovery():
    result = gptoss_runtime.stop_server(WORKING_DIR)
    _gate2_teardown_attempts.append(result)
    return bool(result.get("shutdown_ok")), result, result.get("post_gpu_rows", [])


def _gate2_gpu_rows():
    # Error means unknown, never silently clean.
    return gptoss_runtime.gpu_rows()


def _gate2_make_trial_benchmark(spec):
    trial_bm = _gate2_copy.deepcopy(bm)
    trial_bm.label = f"gate2-{spec['trial_id']}"
    trial_bm.job_dir = _GATE2_ROOT / spec["trial_id"]
    trial_bm.games = [offline_by_id[game_id] for game_id in _gate2_expected_ids]
    trial_bm.n_passes = 1
    trial_bm.game_weights = None
    trial_bm.solver.max_runtime_s_per_game = _gate2_trial_seconds
    trial_bm.solver.analyzer_timeout = 300.0  # T1: smoke previously kept 60s HTTP read timeout
    trial_bm.solver.concurrency = GATE2_CONCURRENCY
    trial_bm.solver.max_actions_per_game = GATE2_ACTION_CAP
    trial_bm.solver.save_request_logs = False
    trial_bm.solver.analyzer_factory = _gate2_make_analyzer_factory(spec, trial_bm.solver)
    return trial_bm


async def _gate2_run_trial(spec, global_soft_epoch, global_hard_epoch):
    global _gate2_hard_guard_triggered
    trial_id = spec["trial_id"]
    _gate2_os.environ["MULTIMODAL_CONTEXT"] = ""
    print("GPTOSS_MODE_ACTIVE image=false scene=true native_tools=true", flush=True)
    _gate2_tool_agent_module._LOCAL_ANALYZER_MAX_OUTPUT = int(spec["cap"])
    _gate2_tool_agent_module._LOCAL_ANALYZER_SEED = int(spec["seed"])
    os.environ["LOCAL_ANALYZER_MAX_OUTPUT"] = str(spec["cap"])
    os.environ["LOCAL_ANALYZER_SEED"] = str(spec["seed"])
    trial_bm = _gate2_make_trial_benchmark(spec)
    _gate2_trial_benchmarks[trial_id] = trial_bm
    trial_started = time.monotonic()
    trial_started_epoch = time.time()
    trial_soft_epoch = min(trial_started_epoch + _gate2_trial_seconds, global_soft_epoch)
    trial_hard_epoch = min(trial_soft_epoch + 60.0, global_hard_epoch)
    if trial_soft_epoch <= trial_started_epoch:
        raise TimeoutError(f"No global runtime remains before trial {trial_id}.")
    print(
        "GATE2_TRIAL_START "
        + json.dumps(
            {
                **spec,
                "games": _gate2_expected_ids,
                "soft_end_epoch": trial_soft_epoch,
                "hard_end_epoch": trial_hard_epoch,
            },
            sort_keys=True,
        ),
        flush=True,
    )
    stop_event = _gate2_asyncio.Event()
    progress_task = _gate2_asyncio.create_task(
        _gate2_progress_loop(trial_id, trial_bm, trial_started, stop_event)
    )
    error = None
    try:
        await _gate2_asyncio.wait_for(
            trial_bm.run(
                soft_end_time=datetime.fromtimestamp(trial_soft_epoch),
                runtime_environment=target,
                minimal_diagnostics=True,
            ),
            timeout=max(1.0, trial_hard_epoch - time.time()),
        )
    except _gate2_asyncio.TimeoutError:
        _gate2_hard_guard_triggered = True
        error = f"Hard runtime guard triggered in {trial_id}."
    except Exception as exc:
        error = repr(exc)
    finally:
        stop_event.set()
        await _gate2_asyncio.gather(progress_task, return_exceptions=True)
        _gate2_flush_trial(trial_id, trial_bm, trial_started)

    rows = _gate2_trial_rows(trial_id)
    gpu_seconds = time.monotonic() - trial_started
    problems = []
    row_ids = [row["game_id"] for row in rows]
    if row_ids != sorted(_gate2_expected_ids):
        problems.append(f"coverage ids={row_ids}")
    if len(rows) != len(_gate2_expected_ids):
        problems.append(f"coverage count={len(rows)}")
    if not all(bool(row.get("terminal")) for row in rows):
        problems.append("non-terminal row")
    if any(row.get("final_score") is None for row in rows):
        problems.append("missing final score")
    if sum(int(row["actions_taken"]) for row in rows) <= 0:
        problems.append("zero actions")
    if error is not None:
        problems.append(error)
    summary = _gate2_summarize_trial(spec, rows, gpu_seconds)
    summary["validation_problems"] = problems
    _gate2_atomic_json(_GATE2_ROOT / trial_id / "summary.json", summary)
    print("GATE2_TRIAL_SUMMARY " + json.dumps(summary, sort_keys=True), flush=True)
    if problems:
        raise RuntimeError(f"Gate 2 trial {trial_id} failed validation: {problems}")
    _gate2_completed_trials.append(trial_id)
    return summary



if GATE2_COMPARISON_SMOKE:
    _gate2_global_soft_epoch = (
        GATE1_SERVER_READY_EPOCH + len(_gate2_trial_specs) * _gate2_trial_seconds + 120.0
    )
    _gate2_global_hard_epoch = _gate2_global_soft_epoch + 60.0
else:
    _gate2_budget = float(target.max_runtime_s)
    _gate2_global_hard_epoch = (
        NOTEBOOK_START_EPOCH + _gate2_budget - GATE2_SAFETY_MARGIN_SECONDS
    )
    _gate2_global_soft_epoch = _gate2_global_hard_epoch - GATE2_SOFT_STOP_GRACE_SECONDS

print(
    "GATE2_DEADLINES "
    + json.dumps(
        {
            "smoke": GATE2_COMPARISON_SMOKE,
            "server_ready_epoch": GATE1_SERVER_READY_EPOCH,
            "startup_seconds": GATE1_SERVER_STARTUP_SECONDS,
            "global_soft_epoch": _gate2_global_soft_epoch,
            "global_hard_epoch": _gate2_global_hard_epoch,
        },
        sort_keys=True,
    ),
    flush=True,
)

try:
    for _gate2_spec in _gate2_trial_specs:
        _gate2_summary = await _gate2_run_trial(
            _gate2_spec,
            _gate2_global_soft_epoch,
            _gate2_global_hard_epoch,
        )
except Exception as exc:
    _gate2_benchmark_error = repr(exc)
    print(f"GATE2_BENCHMARK_EXCEPTION {_gate2_benchmark_error}", flush=True)
finally:
    _gate2_write_progress_index()

try:
    if _gate2_benchmark_error is None:
        _gate2_benchmark_ok = True
except Exception as exc:
    _gate2_benchmark_error = repr(exc)
    _gate2_benchmark_ok = False
    print(f"GATE2_BENCHMARK_VALIDATION_EXCEPTION {_gate2_benchmark_error}", flush=True)
finally:
    _gate2_write_progress_index()


try:
    _gate2_teardown_ok, _gate2_teardown_result, _gate2_post_gpu_rows = (
        _gate2_teardown_with_recovery()
    )
    if not _gate2_teardown_ok:
        _gate2_teardown_error = "bounded terminal gate did not pass after recovery"
except Exception as exc:
    _gate2_teardown_error = repr(exc)
    _gate2_teardown_ok = False
    _gate2_post_gpu_rows = _gate2_gpu_rows()

_gate2_lifecycle = {
    "smoke_mode": GATE2_COMPARISON_SMOKE,
    "benchmark_ok": _gate2_benchmark_ok,
    "benchmark_error": _gate2_benchmark_error,
    "completed_trials": _gate2_completed_trials,
    "expected_trials": [spec["trial_id"] for spec in _gate2_trial_specs],
    "teardown_ok": _gate2_teardown_ok,
    "teardown_error": _gate2_teardown_error,
    "hard_guard_triggered": _gate2_hard_guard_triggered,
    "teardown_attempts": _gate2_teardown_attempts,
    "post_teardown_gpu_rows": _gate2_post_gpu_rows,
    "elapsed_seconds": time.monotonic() - _GATE2_PROGRESS_START,
}
_gate2_atomic_json(_GATE2_ROOT / "lifecycle.json", _gate2_lifecycle)
print(
    "GATE2_LIFECYCLE "
    + json.dumps(
        {
            "benchmark": "ok" if _gate2_benchmark_ok else "failed",
            "benchmark_error": _gate2_benchmark_error,
            "completed_trials": _gate2_completed_trials,
            "teardown": "ok" if _gate2_teardown_ok else "failed",
            "teardown_error": _gate2_teardown_error,
            "hard_guard_triggered": _gate2_hard_guard_triggered,
            "post_teardown_gpu_rows": _gate2_post_gpu_rows,
            "elapsed_seconds": _gate2_lifecycle["elapsed_seconds"],
        },
        sort_keys=True,
    ),
    flush=True,
)


In [ ]:
# A lifecycle pass does not measure an improvement in ARC score.
rows = _gate2_write_progress_index()
problems = []
request_count = response_count = 0
for row in rows:
    label = row["game_id"]
    counts = [int(row.get(k, 0)) for k in (
        "scene_observations", "scene_delivered_messages", "scene_text_only_messages")]
    if not (counts[0] > 0 and counts[0] == counts[1] == counts[2]):
        problems.append(f"{label}: incomplete scene/text delivery {counts}")
    if int(row.get("scene_image_messages", 0)) != 0:
        problems.append(f"{label}: unexpected image payload")
    if int(row.get("llm_calls", 0)) <= 0 or int(row.get("actions_taken", 0)) <= 0:
        problems.append(f"{label}: no instrumented model calls or real actions")
    audit_path = _GATE2_ROOT / "G" / "http" / f"{label}.jsonl"
    events = [json.loads(line) for line in audit_path.read_text().splitlines()] if audit_path.exists() else []
    requests = [e for e in events if e["event"] == "request"]
    responses = [e for e in events if e["event"] == "response"]
    request_count += len(requests)
    response_count += len(responses)
    if len(requests) != int(row.get("llm_calls", 0)) or not responses:
        problems.append(f"{label}: missing provider-boundary request/response evidence")
    for event in requests:
        payload = event["payload"]
        messages = payload.get("messages", [])
        text_only = all(m.get("content") is None or isinstance(m.get("content"), str) for m in messages)
        scene_present = any(m.get("role") == "user" and "SCENE_V1" in (m.get("content") or "") for m in messages)
        tools = [t.get("function", {}).get("name") for t in payload.get("tools", [])]
        if not (text_only and scene_present and "python" in tools
                and payload.get("model") == os.environ["LOCAL_ANALYZER_MODEL_ID"]
                and payload.get("seed") == GATE2_SEEDS[0]
                and payload.get("reasoning_effort") == "high"
                and payload.get("tool_choice") == "auto"
                and payload.get("ignore_eos") is False
                and "top_k" not in payload and "chat_template_kwargs" not in payload):
            problems.append(f"{label}: invalid native provider request {event['sequence']}")
    print("GPTOSS_GAME " + json.dumps(row, sort_keys=True), flush=True)
clean = bool(not problems and _gate2_benchmark_ok and _gate2_teardown_ok
             and not _gate2_hard_guard_triggered and _gate2_completed_trials == ["G"]
             and len(rows) == 1 and all(r.get("terminal") for r in rows)
             and not _gate2_post_gpu_rows)
final = {"smoke": True, "clean": clean, "problems": problems,
         "http_requests": request_count, "http_responses": response_count,
         "benchmark_ok": _gate2_benchmark_ok, "teardown_ok": _gate2_teardown_ok,
         "completed_trials": _gate2_completed_trials, "score_claim": None}
_gate2_atomic_json(WORKING_DIR / "gptoss-smoke-final.json", final)
print("GPTOSS_SMOKE_FINAL " + json.dumps(final, sort_keys=True), flush=True)
print("GPTOSS_SMOKE_OK" if clean else "GPTOSS_SMOKE_FAILED", flush=True)
if not clean:
    raise RuntimeError("GPT-OSS lifecycle smoke failed; results preserved")
